# BTCUSD 5m multi-horizon path research (v9)

Research Colab: learn the causal relationship between historical OHLCV-derived
candle/chart structure and **how the path will behave** at 5m, 10m, 15m, 30m,
1h, and 2h. Continuous geometry is the input; named patterns are an auxiliary
head (not a trading target). Direction is 5-class ATR-normalized
(STRONG_DOWN / DOWN / NEUTRAL / UP / STRONG_UP).

This is **not** next-OHLC prediction, not a live agent, and not a replacement for
the production v6 all-TF trainer until a v9 5m export is validated.

Upload **this notebook only** to Google Colab. Historical OHLCV comes from the
Delta Exchange India public API. Edit repo `.py` files and regenerate:

    python scripts/colab/build_next_candle_research_notebook.py

**Colab setup:** Runtime → Change runtime type → **T4 GPU**.


## 01 Env

In [ ]:
# Colab ships torch/pandas/numpy; install research extras.
!pip install -q --upgrade-strategy only-if-needed pyarrow onnx onnxruntime requests scikit-learn optuna shap matplotlib seaborn

import torch

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — training will run on CPU and be much slower.")
    print("Runtime -> Change runtime type -> select a GPU (T4), then re-run this cell.")


## Feature contract (agent integration)

In [ ]:
"""Constants shared between per-TF Colab training and agent inference."""

from __future__ import annotations

from typing import Any, Dict, Tuple

# Bump when FEATURE_COLS, label heads, or ONNX outputs change (requires retrain).
FEATURE_CONTRACT_VERSION_V6 = "transformer_btcusd_per_tf_features_v6"
FEATURE_CONTRACT_VERSION_V7 = "transformer_btcusd_per_tf_features_v7"
FEATURE_CONTRACT_VERSION_V8 = "transformer_btcusd_per_tf_features_v8"
FEATURE_CONTRACT_VERSION = "transformer_btcusd_per_tf_features_v9"

SUPPORTED_RESOLUTIONS: Tuple[str, ...] = ("5m", "15m", "30m", "1h", "2h")

RESOLUTION_MINUTES: Dict[str, int] = {
    "5m": 5,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
}

TF_KEYS: Tuple[str, ...] = tuple(f"tf_{r}" for r in SUPPORTED_RESOLUTIONS)

HTF_SOURCE_TFS: Tuple[str, ...] = ("15m", "30m", "1h", "2h")
HTF_FEATURE_FIELDS: Tuple[str, ...] = (
    "structure_bias",
    "trend_efficiency",
    "ema21_slope_atr",
    "dist_support_atr",
    "dist_resistance_atr",
    "range_width_atr",
)

NATIVE_STRUCTURE_COLS: Tuple[str, ...] = (
    "hh_count",
    "hl_count",
    "lh_count",
    "ll_count",
    "structure_bias",
    "last_swing_dir",
    "bars_since_swing",
    "swing_amp_atr",
    "unconfirmed_ext_atr",
    "trend_efficiency",
    "displacement_atr",
    "pct_with_trend",
    "price_vs_ema9_atr",
    "price_vs_ema21_atr",
    "price_vs_ema50_atr",
    "price_vs_ema200_atr",
    "ema9_vs_21_atr",
    "ema21_vs_50_atr",
    "ema50_vs_200_atr",
    "ema21_slope_atr",
    "ema50_slope_atr",
    "dist_to_support_atr",
    "dist_to_resistance_atr",
    "support_touch_count",
    "resistance_touch_count",
    "range_width_atr",
    "range_width_pctile",
    "atr_contraction",
    "breakout_size_atr",
    "breakout_vol_ratio",
    "pre_breakout_comp",
    "bars_since_breakout",
    "retest_dist_atr",
    "failed_break",
    "peak_diff_atr",
    "trough_diff_atr",
    "peak_sep_bars",
    "trough_sep_bars",
    "dist_neck_atr",
    "high_slope_atr",
    "low_slope_atr",
    "convergence",
    "width_now_atr",
    "pole_disp_atr",
    "flag_width_atr",
    "flag_slope_atr",
)

HTF_STRUCTURE_COLS: Tuple[str, ...] = tuple(
    f"htf_{tf}_{field}" for tf in HTF_SOURCE_TFS for field in HTF_FEATURE_FIELDS
)

_V4_FEATURE_COLS: Tuple[str, ...] = (
    "ret_1",
    "rv_16",
    "rv_96",
    "ema50_dist_pct",
    "macd_hist",
    "rsi_14",
    "adx_14",
    "obv_z",
    "vol_z",
    "body_ratio",
    "upper_wick_ratio",
    "lower_wick_ratio",
    "close_loc",
    "range_atr",
    "body_atr",
    "gap_atr",
    "inside_bar",
    "outside_bar",
    "engulf_score",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "dist_to_resistance_pct",
    "dist_to_support_pct",
    "funding_rate",
    "oi_z",
    "funding_zscore",
    "funding_mom",
    "funding_rate_roc",
    "oi_change_2",
    "oi_delta_z",
    "oi_price_divergence",
    "oi_acceleration",
    "funding_x_oi",
)

# Native all-TF continuous columns (v4 plus causal structure/geometry).
FEATURE_COLS: Tuple[str, ...] = _V4_FEATURE_COLS + NATIVE_STRUCTURE_COLS

PATH_LABEL_COLS: Tuple[str, ...] = (
    "mfe",
    "mae",
    "future_volatility",
    "trend_strength",
    "drawdown_before_mfe",
    "future_oi_change_pct",
    "future_volume_change_pct",
    "candle_follow_through_atr",
    "structure_delta",
)

CONTINUOUS_LABEL_COLS: Tuple[str, ...] = PATH_LABEL_COLS

PATH_LABEL_HORIZON_BARS: int = 8

# Legacy 8h reference (btcusd_15m_transformer used 32 bars on 15m).
REFERENCE_LABEL_HORIZON_MINUTES: int = 480

DEFAULT_PATH_LABEL_HORIZON_MINUTES: Dict[str, int] = {
    "5m": 240,
    "15m": 240,
    "30m": 480,
    "1h": 480,
    "2h": 480,
}

# Volume change excluded from loss (dominates shared encoder).
DEFAULT_CONTINUOUS_LOSS_WEIGHTS: Dict[str, float] = {
    "mfe": 0.5,
    "mae": 0.5,
    "future_volatility": 1.0,
    "trend_strength": 0.5,
    "drawdown_before_mfe": 0.5,
    "future_oi_change_pct": 0.25,
    "future_volume_change_pct": 0.0,
    "candle_follow_through_atr": 0.5,
    "structure_delta": 0.5,
}

REGIME_NAMES: Dict[int, str] = {
    0: "LOW",
    1: "NORMAL",
    2: "HIGH",
    3: "EXTREME",
}

CANDLE_CLASS_COL = "candle_class_id"
CANDLE_CLASS_CARDINALITY = 13
CANDLE_EMBED_DIM = 8

CANDLE_CLASS_NAMES: Dict[int, str] = {
    0: "FLAT_ZERO_RANGE",
    1: "DOJI_DRAGONFLY",
    2: "DOJI_GRAVESTONE",
    3: "DOJI_STANDARD",
    4: "MARUBOZU_BULL",
    5: "MARUBOZU_BEAR",
    6: "HAMMER_SHAPE",
    7: "INV_HAMMER_SHAPE",
    8: "SPINNING_TOP",
    9: "BELT_HOLD_BULL",
    10: "BELT_HOLD_BEAR",
    11: "STANDARD_BULL",
    12: "STANDARD_BEAR",
}

# Discrete training/inference columns (not continuous ONNX heads).
STRUCTURE_OUTCOME_COL = "future_structure_outcome"
FUTURE_CANDLE_COL = "future_candle_class"
STRUCTURE_OUTCOME_CARDINALITY = 6
N_AUX_CLASS_HEADS = 3  # vol regime + structure outcome + next-bar candle

# v7 next-candle structure (kept for legacy 5m ONNX decode).
NEXT_DIRECTION_COL = "next_direction"
NEXT_BODY_COL = "next_body"
NEXT_WICK_COL = "next_wick"
NEXT_RANGE_COL = "next_range"
CHART_PATTERN_COL = "chart_pattern_id"
VOLUME_STATE_COL = "volume_state"
VOLUME_CONFIRMS_COL = "volume_confirms"
PATTERN_ACTIVE_COL = "pattern_active"
SAMPLE_WEIGHT_COL = "sample_weight"

# v8 wall-clock behavior packets on the 5m grid (bars).
HORIZON_SPECS: Tuple[Tuple[str, int], ...] = (
    ("h5m", 1),
    ("h10m", 2),
    ("h15m", 3),
    ("h30m", 6),
    ("h1h", 12),
    ("h2h", 24),
)
HORIZON_KEYS: Tuple[str, ...] = tuple(key for key, _ in HORIZON_SPECS)
HORIZON_BARS_5M: Tuple[int, ...] = tuple(bars for _, bars in HORIZON_SPECS)
N_HORIZONS: int = len(HORIZON_SPECS)
MAX_V8_HORIZON_BARS: int = max(HORIZON_BARS_5M)
HORIZON_CONTINUOUS_FIELDS: Tuple[str, ...] = ("mfe", "mae", "vol", "trend_strength")
HORIZON_DIR_COLS: Tuple[str, ...] = tuple(f"{key}_dir" for key in HORIZON_KEYS)
HORIZON_STRUCTURE_COLS: Tuple[str, ...] = tuple(
    f"{key}_structure" for key in HORIZON_KEYS
)
V8_CONTINUOUS_LABEL_COLS: Tuple[str, ...] = tuple(
    f"{key}_{field}"
    for key in HORIZON_KEYS
    for field in HORIZON_CONTINUOUS_FIELDS
)
HORIZON_DIR_COLS_V7: Tuple[str, ...] = (
    "horizon_t1_dir",
    "horizon_t3_dir",
    "horizon_t6_dir",
    "horizon_t12_dir",
    "horizon_t24_dir",
)

NEXT_DIRECTION_CARDINALITY_V8 = 3
NEXT_DIRECTION_CARDINALITY = 5
NEXT_BODY_CARDINALITY = 3
NEXT_WICK_CARDINALITY = 4
NEXT_RANGE_CARDINALITY = 3
VOLUME_STATE_CARDINALITY = 3
CHART_PATTERN_CARDINALITY = 9

NEXT_DIRECTION_NAMES_V8: Dict[int, str] = {0: "BEARISH", 1: "NEUTRAL", 2: "BULLISH"}
NEXT_DIRECTION_NAMES: Dict[int, str] = {
    0: "STRONG_DOWN",
    1: "DOWN",
    2: "NEUTRAL",
    3: "UP",
    4: "STRONG_UP",
}
BULLISH_DIRECTION_NAMES = frozenset({"UP", "STRONG_UP", "BULLISH"})
BEARISH_DIRECTION_NAMES = frozenset({"DOWN", "STRONG_DOWN", "BEARISH"})
NEUTRAL_DIRECTION_NAMES = frozenset({"NEUTRAL"})
NEXT_BODY_NAMES: Dict[int, str] = {0: "SMALL", 1: "MEDIUM", 2: "LARGE"}
NEXT_WICK_NAMES: Dict[int, str] = {
    0: "BALANCED",
    1: "UPPER_REJECTION",
    2: "LOWER_REJECTION",
    3: "BOTH_REJECTION",
}
NEXT_RANGE_NAMES: Dict[int, str] = {0: "COMPRESSED", 1: "NORMAL", 2: "EXPANDED"}
VOLUME_STATE_NAMES: Dict[int, str] = {0: "DRY", 1: "NORMAL", 2: "EXPANSION"}
CHART_PATTERN_NAMES: Dict[int, str] = {
    0: "NONE",
    1: "FLAG_BULL",
    2: "FLAG_BEAR",
    3: "TRIANGLE",
    4: "DOUBLE_TOP",
    5: "DOUBLE_BOTTOM",
    6: "CHANNEL",
    7: "BREAKOUT",
    8: "FAILED_BREAK",
}

VOL_Z_EXPANSION = 0.5
VOL_Z_DRY = -0.5
BREAKOUT_VOL_CONFIRM = 1.2
HORIZON_DIR_ATR_WEAK = 0.5
HORIZON_DIR_ATR_STRONG = 2.0
HORIZON_DIR_ATR_DEADZONE = HORIZON_DIR_ATR_WEAK

V7_STRUCTURE_LOSS_WEIGHTS: Dict[str, float] = {
    "direction": 1.0,
    "body": 0.5,
    "wick": 0.5,
    "range": 0.5,
    "pattern": 0.5,
    "path": 1.0,
    "volume_state": 0.25,
    "pattern_validates": 0.25,
    "horizon": 0.35,
}

V8_STRUCTURE_LOSS_WEIGHTS: Dict[str, float] = {
    "path": 1.0,
    "direction": 1.0,
    "structure": 0.5,
    "volume_state": 0.25,
}
V9_STRUCTURE_LOSS_WEIGHTS: Dict[str, float] = {
    "path": 1.0,
    "direction": 1.0,
    "structure": 0.5,
    "volume_state": 0.25,
    "pattern": 0.25,
}

V7_CONTINUOUS_LOSS_WEIGHTS: Dict[str, float] = {
    "mfe": 0.5,
    "mae": 0.5,
    "future_volatility": 1.0,
    "trend_strength": 0.5,
    "drawdown_before_mfe": 0.5,
    "future_oi_change_pct": 0.25,
    "future_volume_change_pct": 0.25,
    "candle_follow_through_atr": 0.5,
    "structure_delta": 0.5,
}

STRUCTURE_OUTCOME_NAMES: Dict[int, str] = {
    0: "RANGE",
    1: "CONTINUATION_LONG",
    2: "CONTINUATION_SHORT",
    3: "BREAKOUT",
    4: "FAILED_BREAK",
    5: "REVERSAL",
}

ONNX_OUTPUT_NAMES_V6: Tuple[str, ...] = (
    "continuous_pred",
    "regime_logits",
    "structure_outcome_logits",
    "future_candle_logits",
)
# Live 15m–2h bundles still export v6 heads. 5m research is v8.
ONNX_OUTPUT_NAMES: Tuple[str, ...] = ONNX_OUTPUT_NAMES_V6

ONNX_OUTPUT_NAMES_V7_BASE: Tuple[str, ...] = (
    "next_direction_logits",
    "next_body_logits",
    "next_wick_logits",
    "next_range_logits",
    "next_pattern_logits",
    "continuous_pred",
    "volume_state_logits",
    "pattern_validates_logit",
)
ONNX_OUTPUT_NAMES_V7_HORIZONS: Tuple[str, ...] = (
    "horizon_t3_dir_logits",
    "horizon_t6_dir_logits",
    "horizon_t12_dir_logits",
    "horizon_t24_dir_logits",
)
ONNX_OUTPUT_NAMES_V7: Tuple[str, ...] = (
    ONNX_OUTPUT_NAMES_V7_BASE + ONNX_OUTPUT_NAMES_V7_HORIZONS
)
ONNX_OUTPUT_NAMES_V8_DIR: Tuple[str, ...] = tuple(
    f"{key}_dir_logits" for key in HORIZON_KEYS
)
ONNX_OUTPUT_NAMES_V8_STRUCTURE: Tuple[str, ...] = tuple(
    f"{key}_structure_logits" for key in HORIZON_KEYS
)
ONNX_OUTPUT_NAMES_V8: Tuple[str, ...] = (
    ONNX_OUTPUT_NAMES_V8_DIR
    + ONNX_OUTPUT_NAMES_V8_STRUCTURE
    + ("continuous_pred", "volume_state_logits")
)
ONNX_OUTPUT_NAMES_V9: Tuple[str, ...] = ONNX_OUTPUT_NAMES_V8 + ("chart_pattern_logits",)

# Next-bar candle families for 5m timing (not model classes).
CANDLE_FAMILY_DOJI = frozenset({0, 1, 2, 3, 8})
CANDLE_FAMILY_BULL = frozenset({4, 6, 9, 11})
CANDLE_FAMILY_BEAR = frozenset({5, 7, 10, 12})

# Minimum test-set correlation for future_volatility before ONNX export.
MIN_EXPORT_VOL_CORR: Dict[str, float] = {
    "5m": 0.10,
    "15m": 0.10,
    "30m": 0.10,
    "1h": 0.08,
    "2h": 0.08,
}

PROMOTION_VOL_CORR: float = 0.15
PROMOTION_REGIME_ACCURACY: float = 0.35

EXPORT_QUALITY_DISCLAIMER = (
    "Sanity gates detect broken exports, not trading edge. "
    "Promotion tier targets are informational."
)

TRANSFORMER_METADATA_FILENAME = "metadata_transformer.json"
TRANSFORMER_FEATURE_CONFIG_FILENAME = "feature_config.json"

def candle_family_from_class(class_id: int) -> str:
    """Map a candle class id to bull / bear / doji for timing modifiers."""
    cid = int(class_id)
    if cid in CANDLE_FAMILY_BULL:
        return "bull"
    if cid in CANDLE_FAMILY_BEAR:
        return "bear"
    return "doji"

def compute_path_edge(mfe: float, mae: float) -> float:
    """Directional edge from predicted path asymmetry (MFE minus MAE).

    Alias of :func:`compute_long_edge` kept for train/serve telemetry compatibility.
    """
    return compute_long_edge(mfe, mae)

def compute_long_edge(mfe: float, mae: float) -> float:
    """Long-side path edge: upside (MFE) minus downside (MAE)."""
    return float(mfe) - float(mae)

def compute_short_edge(mfe: float, mae: float) -> float:
    """Short-side path edge: downside (MAE) minus upside (MFE)."""
    return float(mae) - float(mfe)

def path_favorable_adverse(
    mfe: float,
    mae: float,
    *,
    side: str,
) -> Tuple[float, float]:
    """Return (favorable_pct, adverse_pct) for bracket sizing.

    Labels are long-centric (MFE = upside, MAE = downside). For shorts, favorable
    excursion is downside (MAE) and adverse is upside (MFE).
    """
    s = str(side or "BUY").strip().upper()
    if s in ("SELL", "SHORT", "STRONG_SELL"):
        return float(mae), float(mfe)
    return float(mfe), float(mae)

def model_family_for_resolution(resolution: str) -> str:
    """Canonical model_family string for a TF bundle."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    return f"jacksparrow_transformer_btcusd_{res}"

def onnx_filename_for_resolution(resolution: str) -> str:
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    return f"btcusd_{res}_transformer.onnx"

def bundle_dir_name(resolution: str) -> str:
    res = resolution.strip().lower()
    return f"JackSparrow_Transformer_BTCUSD_{res}"

def horizon_bars_for_wall_minutes(wall_minutes: int, resolution_minutes: int) -> int:
    """Convert wall-clock minutes to native-TF bar count."""
    return max(1, int(round(int(wall_minutes) / resolution_minutes)))

def label_horizon_bars_for_resolution(resolution_minutes: int) -> int:
    """Legacy helper: 8h wall-clock in bars for a TF grid."""
    return horizon_bars_for_wall_minutes(REFERENCE_LABEL_HORIZON_MINUTES, resolution_minutes)

def path_label_horizon_bars_for_resolution(resolution: str) -> int:
    """Path-label forward window in bars for a TF."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    minutes = RESOLUTION_MINUTES[res]
    wall = DEFAULT_PATH_LABEL_HORIZON_MINUTES.get(res, REFERENCE_LABEL_HORIZON_MINUTES)
    return horizon_bars_for_wall_minutes(wall, minutes)

def continuous_loss_weights_for_resolution(resolution: str) -> Tuple[float, ...]:
    """Per-head loss weights aligned with CONTINUOUS_LABEL_COLS (0 = no gradient)."""
    weights = dict(DEFAULT_CONTINUOUS_LOSS_WEIGHTS)
    return tuple(float(weights.get(col, 1.0)) for col in CONTINUOUS_LABEL_COLS)

def default_training_config(resolution: str) -> Dict[str, Any]:
    """Default Colab training config for a single TF model."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    minutes = RESOLUTION_MINUTES[res]
    path_horizon = path_label_horizon_bars_for_resolution(res)
    return {
        "symbol": "BTCUSD",
        "resolution": res,
        "resolution_minutes": minutes,
        "history_days": 900,
        "base_url": "https://api.india.delta.exchange",
        "atr_period": 14,
        "path_label_horizon_bars": path_horizon,
        "label_horizon_minutes_path": DEFAULT_PATH_LABEL_HORIZON_MINUTES.get(
            res, REFERENCE_LABEL_HORIZON_MINUTES
        ),
        "continuous_loss_weights": list(continuous_loss_weights_for_resolution(res)),
        "mae_floor_atr_mult": 0.25,
        "vol_regime_quantiles": [0.25, 0.5, 0.75],
        "window_len": 128,
        "stride": 8,
        "train_frac": 0.65,
        "val_frac": 0.15,
        "embargo_bars": path_horizon,
        "batch_size": 128,
        "epochs": 120,
        "lr": 1e-4,
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "dropout": 0.25,
        "weight_decay": 1e-2,
        "early_stop_patience": 12,
        "early_stopping_enabled": True,
        "min_derivatives_coverage": 0.5,
        "derivatives_coverage_warn": 0.9,
        "min_export_vol_corr": MIN_EXPORT_VOL_CORR.get(res, 0.08),
        "default_threshold": 0.005,
        "seed": 42,
    }

def max_label_horizon_bars(
    path_label_horizon_bars: int = PATH_LABEL_HORIZON_BARS,
) -> int:
    """Maximum forward bars across all training labels."""
    return int(path_label_horizon_bars)

def scale_period(period: int, resolution_minutes: int, *, base_minutes: int = 5) -> int:
    """Scale indicator lookback to preserve wall-clock semantics across TFs."""
    return max(1, int(round(period * resolution_minutes / base_minutes)))

def feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:
    """Continuous feature columns for a TF bundle (5m appends closed HTF context)."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    if res == "5m":
        return FEATURE_COLS + HTF_STRUCTURE_COLS
    return FEATURE_COLS

def v7_feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:
    """Input columns: native features plus causal chart_pattern_id."""
    return feature_cols_for_resolution(resolution) + (CHART_PATTERN_COL,)

def v8_feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:
    """v8 5m input columns (native features plus causal chart_pattern_id)."""
    return v7_feature_cols_for_resolution(resolution)

def v9_feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:
    """v9 5m input columns: geometry only; chart_pattern_id is a target head."""
    return feature_cols_for_resolution(resolution)

def onnx_output_names_for_contract(
    contract_version: str,
    *,
    resolution: str = "5m",
) -> Tuple[str, ...]:
    """ONNX head names for a bundle contract."""
    ver = str(contract_version or "").strip()
    res = resolution.strip().lower()
    if ver == FEATURE_CONTRACT_VERSION:
        if res == "5m":
            return ONNX_OUTPUT_NAMES_V9
        return ONNX_OUTPUT_NAMES_V6
    if ver == FEATURE_CONTRACT_VERSION_V8:
        if res == "5m":
            return ONNX_OUTPUT_NAMES_V8
        return ONNX_OUTPUT_NAMES_V6
    if ver == FEATURE_CONTRACT_VERSION_V7:
        if res == "5m":
            return ONNX_OUTPUT_NAMES_V7
        return ONNX_OUTPUT_NAMES_V7_BASE
    return ONNX_OUTPUT_NAMES_V6

def v8_future_leak_cols() -> frozenset:
    """Label/target columns that must never appear in 5m v8 model inputs."""
    leaked = set(V8_CONTINUOUS_LABEL_COLS)
    leaked.update(HORIZON_DIR_COLS)
    leaked.update(HORIZON_STRUCTURE_COLS)
    leaked.update(
        {
            VOLUME_STATE_COL,
            NEXT_DIRECTION_COL,
            NEXT_BODY_COL,
            NEXT_WICK_COL,
            NEXT_RANGE_COL,
            FUTURE_CANDLE_COL,
            "mfe",
            "mae",
            "future_volatility",
            "pattern_validates",
            *HORIZON_DIR_COLS_V7,
        }
    )
    return frozenset(leaked)

def expected_direction_from_chart_pattern(pattern_id: int) -> int:
    """Map chart pattern to expected next-candle direction (0/1/2). Neutral if none."""
    pid = int(pattern_id)
    if pid in (1, 5, 7):  # FLAG_BULL, DOUBLE_BOTTOM, BREAKOUT (unsigned handled elsewhere)
        return 2
    if pid in (2, 4, 8):  # FLAG_BEAR, DOUBLE_TOP, FAILED_BREAK
        return 0
    return 1

def default_research_config() -> Dict[str, Any]:
    """Colab research-pipeline defaults (5m multi-horizon path)."""
    return {
        "symbol": "BTCUSD",
        "resolution": "5m",
        "resolution_minutes": 5,
        "base_timeframe": "5m",
        "context_timeframes": ["15m", "30m", "1h", "2h"],
        "sequence_length": 64,
        "prediction_horizon": 1,
        "horizon_bars": list(HORIZON_BARS_5M),
        "train_ratio": 0.70,
        "validation_ratio": 0.15,
        "test_ratio": 0.15,
        "batch_size": 256,
        "epochs": 50,
        "learning_rate": 1e-4,
        "weight_decay": 1e-4,
        "dropout": 0.15,
        "early_stopping_patience": 8,
        "seed": 42,
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "stride": 4,
        "scaler_mode": "train_fit",
        "gate_chart_volume": False,
        "path_label_horizon_bars": MAX_V8_HORIZON_BARS,
        "embargo_bars": MAX_V8_HORIZON_BARS,
        "mae_floor_atr_mult": 0.25,
        "atr_period": 14,
        "history_days": 900,
        "base_url": "https://api.india.delta.exchange",
        "loss_weights": dict(V9_STRUCTURE_LOSS_WEIGHTS),
        "run_optuna": False,
        "optuna_trials": 0,
        "run_shap": False,
        "run_ablations": False,
        "run_walk_forward": False,
        "walk_forward_folds": 3,
    }

def ablation_feature_groups(resolution: str = "5m") -> Dict[str, Tuple[str, ...]]:
    """Nested feature sets A-F for out-of-sample ablation."""
    all_cols = v9_feature_cols_for_resolution(resolution)
    ohlcv = ("ret_1", "rv_16", "rv_96", "hour_sin", "hour_cos", "dow_sin", "dow_cos")
    geometry = ohlcv + (
        "body_ratio",
        "upper_wick_ratio",
        "lower_wick_ratio",
        "close_loc",
        "range_atr",
        "body_atr",
        "gap_atr",
        "inside_bar",
        "outside_bar",
        "engulf_score",
    )
    trend = geometry + (
        "ema50_dist_pct",
        "macd_hist",
        "rsi_14",
        "adx_14",
        "price_vs_ema9_atr",
        "price_vs_ema21_atr",
        "price_vs_ema50_atr",
        "price_vs_ema200_atr",
        "ema9_vs_21_atr",
        "ema21_vs_50_atr",
        "ema50_vs_200_atr",
        "ema21_slope_atr",
        "ema50_slope_atr",
        "trend_efficiency",
        "displacement_atr",
        "pct_with_trend",
    )
    structure = trend + (
        "hh_count",
        "hl_count",
        "lh_count",
        "ll_count",
        "structure_bias",
        "last_swing_dir",
        "bars_since_swing",
        "swing_amp_atr",
        "dist_to_support_atr",
        "dist_to_resistance_atr",
        "support_touch_count",
        "resistance_touch_count",
        "range_width_atr",
        "dist_to_resistance_pct",
        "dist_to_support_pct",
    )
    chart = structure + (
        "peak_diff_atr",
        "trough_diff_atr",
        "peak_sep_bars",
        "trough_sep_bars",
        "dist_neck_atr",
        "high_slope_atr",
        "low_slope_atr",
        "convergence",
        "width_now_atr",
        "pole_disp_atr",
        "flag_width_atr",
        "flag_slope_atr",
        "breakout_size_atr",
        "breakout_vol_ratio",
        "pre_breakout_comp",
        "bars_since_breakout",
        "retest_dist_atr",
        "failed_break",
    )
    return {
        "A": tuple(c for c in ohlcv if c in all_cols),
        "B": tuple(c for c in geometry if c in all_cols),
        "C": tuple(c for c in trend if c in all_cols),
        "D": tuple(c for c in structure if c in all_cols),
        "E": tuple(c for c in chart if c in all_cols),
        "F": all_cols,
    }


## Derivatives features

In [ ]:
"""Funding and OI derivative features scaled per resolution."""

from __future__ import annotations

import numpy as np
import pandas as pd

_EPS = 1e-9

def compute_funding_derivatives(
    fund_rate: pd.Series,
    ret_2: pd.Series,
    *,
    resolution_minutes: int,
) -> pd.DataFrame:
    """Funding z-score, momentum, and rate-of-change on native TF bars."""
    z_window = scale_period(16, resolution_minutes)
    roc_diff = max(1, scale_period(1, resolution_minutes))
    fz_mean = fund_rate.rolling(z_window, min_periods=5).mean()
    fz_std = fund_rate.rolling(z_window, min_periods=5).std().replace(0, _EPS)
    funding_zscore = ((fund_rate - fz_mean) / fz_std).fillna(0.0).clip(-4.0, 4.0)
    funding_mom = funding_zscore * ret_2
    diff = fund_rate.diff(roc_diff)
    roc_mu = diff.rolling(z_window, min_periods=5).mean()
    roc_std = diff.rolling(z_window, min_periods=5).std().replace(0, _EPS)
    funding_rate_roc = ((diff - roc_mu) / roc_std).fillna(0.0).clip(-4.0, 4.0)
    return pd.DataFrame(
        {
            "funding_zscore": funding_zscore,
            "funding_mom": funding_mom.fillna(0.0),
            "funding_rate_roc": funding_rate_roc,
        }
    )

def compute_oi_derivatives(
    primary: pd.DataFrame,
    *,
    resolution_minutes: int,
) -> pd.DataFrame:
    """OI-derived features on native TF bars."""
    n = len(primary)
    zero = pd.DataFrame(
        {
            "oi_change_2": np.zeros(n),
            "oi_delta_z": np.zeros(n),
            "oi_price_divergence": np.zeros(n),
            "oi_acceleration": np.zeros(n),
            "oi_zscore": np.zeros(n),
        },
        index=primary.index,
    )
    if "open_interest" not in primary.columns:
        return zero

    z_window = scale_period(16, resolution_minutes)
    change_window = max(1, scale_period(2, resolution_minutes))

    oi_s = primary["open_interest"].astype(float)
    if oi_s.notna().sum() == 0 or float(oi_s.max()) < _EPS:
        return zero

    oi_mu = oi_s.rolling(z_window, min_periods=max(2, z_window // 4)).mean()
    oi_std = oi_s.rolling(z_window, min_periods=max(2, z_window // 4)).std().clip(
        lower=_EPS
    )
    oi_zscore = ((oi_s - oi_mu) / oi_std).fillna(0.0).clip(-4.0, 4.0)

    oi_lagged = oi_s.shift(change_window)
    oi_change_2 = (
        ((oi_s - oi_lagged) / (oi_lagged.abs() + _EPS)).fillna(0.0).clip(-0.05, 0.05)
    )

    close_s = primary["close"].astype(float)
    close_lagged = close_s.shift(change_window)
    ret_2 = ((close_s - close_lagged) / (close_lagged.abs() + _EPS)).fillna(0.0)
    oi_price_divergence = (
        np.sign(oi_change_2.values) * -np.sign(ret_2.values)
    ).astype(np.float32)
    oi_acceleration = oi_change_2.diff().fillna(0.0).clip(-0.02, 0.02)

    oi_delta_1 = oi_s.diff(1).fillna(0.0)
    oi_delta_mu = oi_delta_1.rolling(z_window, min_periods=max(2, z_window // 4)).mean()
    oi_delta_std = oi_delta_1.rolling(z_window, min_periods=max(2, z_window // 4)).std().clip(
        lower=_EPS
    )
    oi_delta_z = ((oi_delta_1 - oi_delta_mu) / oi_delta_std).fillna(0.0).clip(-4.0, 4.0)

    return pd.DataFrame(
        {
            "oi_zscore": oi_zscore,
            "oi_change_2": oi_change_2,
            "oi_price_divergence": pd.Series(oi_price_divergence).fillna(0.0),
            "oi_acceleration": oi_acceleration,
            "oi_delta_z": oi_delta_z,
        },
        index=primary.index,
    ).replace([np.inf, -np.inf], 0.0).fillna(0.0)


## Market structure

In [ ]:
"""Causal OHLCV market-structure and chart-geometry features.

All values at bar t use information available at or before t. Swings are ATR
ZigZag confirmations (not future-looking fractals). Donchian/breakout ranges
exclude bar t. Higher-TF context is merged from fully closed resampled bars.
"""

from __future__ import annotations

from collections import deque
from typing import Deque, List, Tuple

import numpy as np
import pandas as pd

_EPS = 1e-9
_ATR_CLIP = 8.0
_SLOPE_CLIP = 2.0
_ZZ_TAU = 1.5
_ZZ_EVENT_M = 6
_ZZ_EPS_ATR = 0.15
_SWING_K = 20
_ER_N = 32
_SLOPE_LAG = 8
_DONCHIAN_N = 32
_RANGE_PCTILE_W = 256
_TOUCH_W = 64
_SR_DELTA_ATR = 0.5
_POLE_BARS = 12
_FLAG_BARS = 16
_BREAKOUT_THRESH = 0.25
_FAILED_BREAK_BARS = 8
_CHANNEL_K = 3
_ATR_SHORT = 14
_ATR_LONG = 56
_HTF_NATIVE_MINUTES = 5

_HTF_NATIVE_MAP = {
    "structure_bias": "structure_bias",
    "trend_efficiency": "trend_efficiency",
    "ema21_slope_atr": "ema21_slope_atr",
    "dist_support_atr": "dist_to_support_atr",
    "dist_resistance_atr": "dist_to_resistance_atr",
    "range_width_atr": "range_width_atr",
}

def _safe_atr(atr: np.ndarray, close: np.ndarray, i: int) -> float:
    val = float(atr[i]) if np.isfinite(atr[i]) else float("nan")
    if not np.isfinite(val) or val <= 0:
        c = float(close[i]) if np.isfinite(close[i]) else 0.0
        return max(abs(c) * 0.01, _EPS)
    return val

def _clip_atr(val: float) -> float:
    if not np.isfinite(val):
        return 0.0
    return float(np.clip(val, -_ATR_CLIP, _ATR_CLIP))

def _ols_slope(x: np.ndarray, y: np.ndarray) -> float:
    n = float(len(x))
    if n < 2:
        return 0.0
    sx = float(x.sum())
    sy = float(y.sum())
    sxy = float((x * y).sum())
    sx2 = float((x * x).sum())
    den = n * sx2 - sx * sx
    if abs(den) < 1e-12:
        return 0.0
    return float((n * sxy - sx * sy) / den)

def _ensure_atr(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    out = df.copy()
    if "atr" in out.columns and out["atr"].notna().any():
        return out
    prev_close = out["close"].shift(1)
    tr = pd.concat(
        [
            out["high"] - out["low"],
            (out["high"] - prev_close).abs(),
            (out["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    out["atr"] = tr.rolling(period, min_periods=1).mean()
    return out

def _vectorized_trend_and_ma(out: pd.DataFrame) -> pd.DataFrame:
    """ER, displacement, MA distances/slopes, Donchian compression, flag geometry."""
    close = out["close"].astype(float)
    high = out["high"].astype(float)
    low = out["low"].astype(float)
    volume = out["volume"].astype(float) if "volume" in out.columns else pd.Series(
        0.0, index=out.index
    )
    atr = out["atr"].astype(float) if "atr" in out.columns else pd.Series(
        close.abs() * 0.01, index=out.index
    )
    atr_safe = atr.replace(0, np.nan).fillna(close.abs() * 0.01 + _EPS)

    abs_move = (close - close.shift(_ER_N)).abs()
    path = close.diff().abs().rolling(_ER_N, min_periods=2).sum()
    out["trend_efficiency"] = (abs_move / (path + _EPS)).clip(0.0, 1.0).fillna(0.0)
    out["displacement_atr"] = (abs_move / (atr_safe + _EPS)).clip(
        0.0, _ATR_CLIP
    ).fillna(0.0)

    window_dir = np.sign(close - close.shift(_ER_N))
    up_frac = (close.diff() > 0).astype(float).rolling(_ER_N, min_periods=1).mean()
    pct = np.where(window_dir >= 0, up_frac, 1.0 - up_frac)
    out["pct_with_trend"] = pd.Series(pct, index=out.index).fillna(0.0)

    ema9 = close.ewm(span=9, adjust=False).mean()
    ema21 = close.ewm(span=21, adjust=False).mean()
    ema50 = close.ewm(span=50, adjust=False).mean()
    ema200 = close.ewm(span=200, adjust=False).mean()
    out["price_vs_ema9_atr"] = ((close - ema9) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["price_vs_ema21_atr"] = ((close - ema21) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["price_vs_ema50_atr"] = ((close - ema50) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["price_vs_ema200_atr"] = ((close - ema200) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["ema9_vs_21_atr"] = ((ema9 - ema21) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["ema21_vs_50_atr"] = ((ema21 - ema50) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["ema50_vs_200_atr"] = ((ema50 - ema200) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    lag = float(_SLOPE_LAG)
    out["ema21_slope_atr"] = (
        (ema21 - ema21.shift(_SLOPE_LAG)) / (atr_safe * lag + _EPS)
    ).clip(-_SLOPE_CLIP, _SLOPE_CLIP).fillna(0.0)
    out["ema50_slope_atr"] = (
        (ema50 - ema50.shift(_SLOPE_LAG)) / (atr_safe * lag + _EPS)
    ).clip(-_SLOPE_CLIP, _SLOPE_CLIP).fillna(0.0)

    prior_high = high.shift(1)
    prior_low = low.shift(1)
    donch_high = prior_high.rolling(_DONCHIAN_N, min_periods=1).max()
    donch_low = prior_low.rolling(_DONCHIAN_N, min_periods=1).min()
    range_width_atr = ((donch_high - donch_low) / (atr_safe + _EPS)).clip(
        0.0, _ATR_CLIP
    )
    out["range_width_atr"] = range_width_atr.fillna(0.0)
    out["range_width_pctile"] = range_width_atr.rolling(
        _RANGE_PCTILE_W, min_periods=8
    ).rank(pct=True).fillna(0.0)
    atr_short = atr_safe.rolling(_ATR_SHORT, min_periods=1).mean()
    atr_long = atr_safe.rolling(_ATR_LONG, min_periods=1).mean()
    out["atr_contraction"] = (atr_short / (atr_long + _EPS)).clip(0.0, _ATR_CLIP).fillna(
        1.0
    )

    out["breakout_size_atr"] = ((close - donch_high) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    ).fillna(0.0)
    vol_mean = volume.shift(1).rolling(_DONCHIAN_N, min_periods=1).mean()
    out["breakout_vol_ratio"] = (volume / (vol_mean + _EPS)).clip(0.0, _ATR_CLIP).fillna(
        0.0
    )
    out["pre_breakout_comp"] = out["range_width_pctile"].shift(1).fillna(0.0)

    pole_end = close.shift(_FLAG_BARS)
    pole_start = close.shift(_POLE_BARS + _FLAG_BARS)
    out["pole_disp_atr"] = ((pole_end - pole_start) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    ).fillna(0.0)
    flag_high = high.rolling(_FLAG_BARS, min_periods=1).max()
    flag_low = low.rolling(_FLAG_BARS, min_periods=1).min()
    out["flag_width_atr"] = ((flag_high - flag_low) / (atr_safe + _EPS)).clip(
        0.0, _ATR_CLIP
    ).fillna(0.0)
    flag_lag = float(_FLAG_BARS)
    out["flag_slope_atr"] = (
        (close - close.shift(_FLAG_BARS)) / (atr_safe * flag_lag + _EPS)
    ).clip(-_SLOPE_CLIP, _SLOPE_CLIP).fillna(0.0)
    return out

def _zigzag_structure_loop(out: pd.DataFrame) -> pd.DataFrame:
    """Single-pass ATR ZigZag: HH/HL counts, S/R, breakout events, pattern geometry."""
    n = len(out)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    close = out["close"].to_numpy(dtype=np.float64)
    atr = out["atr"].to_numpy(dtype=np.float64) if "atr" in out.columns else np.full(
        n, np.nan
    )
    prior_high = np.roll(high, 1)
    prior_high[0] = high[0]
    prior_low = np.roll(low, 1)
    prior_low[0] = low[0]
    donch_h = np.empty(n, dtype=np.float64)
    donch_l = np.empty(n, dtype=np.float64)
    max_q: Deque[int] = deque()
    min_q: Deque[int] = deque()
    for t in range(n):
        left = t - _DONCHIAN_N
        while max_q and max_q[0] <= left:
            max_q.popleft()
        while min_q and min_q[0] <= left:
            min_q.popleft()
        src_h = prior_high[t]
        src_l = prior_low[t]
        while max_q and prior_high[max_q[-1]] <= src_h:
            max_q.pop()
        while min_q and prior_low[min_q[-1]] >= src_l:
            min_q.pop()
        max_q.append(t)
        min_q.append(t)
        donch_h[t] = prior_high[max_q[0]]
        donch_l[t] = prior_low[min_q[0]]

    hh_count = np.zeros(n, dtype=np.float64)
    hl_count = np.zeros(n, dtype=np.float64)
    lh_count = np.zeros(n, dtype=np.float64)
    ll_count = np.zeros(n, dtype=np.float64)
    structure_bias = np.zeros(n, dtype=np.float64)
    last_swing_dir = np.zeros(n, dtype=np.float64)
    bars_since_swing = np.zeros(n, dtype=np.float64)
    swing_amp_atr = np.zeros(n, dtype=np.float64)
    unconfirmed_ext_atr = np.zeros(n, dtype=np.float64)
    dist_to_support_atr = np.zeros(n, dtype=np.float64)
    dist_to_resistance_atr = np.zeros(n, dtype=np.float64)
    support_touch = np.zeros(n, dtype=np.float64)
    resistance_touch = np.zeros(n, dtype=np.float64)
    bars_since_breakout = np.zeros(n, dtype=np.float64)
    retest_dist_atr = np.zeros(n, dtype=np.float64)
    failed_break = np.zeros(n, dtype=np.float64)
    peak_diff_atr = np.zeros(n, dtype=np.float64)
    trough_diff_atr = np.zeros(n, dtype=np.float64)
    peak_sep_bars = np.zeros(n, dtype=np.float64)
    trough_sep_bars = np.zeros(n, dtype=np.float64)
    dist_neck_atr = np.zeros(n, dtype=np.float64)
    high_slope_atr = np.zeros(n, dtype=np.float64)
    low_slope_atr = np.zeros(n, dtype=np.float64)
    convergence = np.zeros(n, dtype=np.float64)
    width_now_atr = np.zeros(n, dtype=np.float64)

    direction = 1
    e_idx = 0
    e_price = high[0] if n else 0.0
    last_high_price = float("nan")
    last_low_price = float("nan")
    last_high_idx = -1
    last_low_idx = -1
    last_swing_idx = -1
    last_dir = 0.0
    highs: List[Tuple[int, float]] = []
    lows: List[Tuple[int, float]] = []
    events: Deque[str] = deque(maxlen=_ZZ_EVENT_M)
    last_bo_idx = -1
    last_bo_level = float("nan")
    last_bo_side = 0

    for t in range(n):
        atr_t = _safe_atr(atr, close, t)
        thresh = _ZZ_TAU * atr_t
        eps = _ZZ_EPS_ATR * atr_t

        if direction == 1:
            if high[t] >= e_price:
                e_price = high[t]
                e_idx = t
            elif (e_price - low[t]) >= thresh:
                highs.append((e_idx, e_price))
                if len(highs) > _SWING_K:
                    highs = highs[-_SWING_K:]
                if np.isfinite(last_high_price):
                    if e_price > last_high_price + eps:
                        events.append("HH")
                    elif e_price < last_high_price - eps:
                        events.append("LH")
                last_high_price = e_price
                last_high_idx = e_idx
                last_swing_idx = e_idx
                last_dir = 1.0
                direction = -1
                e_price = low[t]
                e_idx = t
        else:
            if low[t] <= e_price:
                e_price = low[t]
                e_idx = t
            elif (high[t] - e_price) >= thresh:
                lows.append((e_idx, e_price))
                if len(lows) > _SWING_K:
                    lows = lows[-_SWING_K:]
                if np.isfinite(last_low_price):
                    if e_price > last_low_price + eps:
                        events.append("HL")
                    elif e_price < last_low_price - eps:
                        events.append("LL")
                last_low_price = e_price
                last_low_idx = e_idx
                last_swing_idx = e_idx
                last_dir = -1.0
                direction = 1
                e_price = high[t]
                e_idx = t

        n_hh = sum(1 for e in events if e == "HH")
        n_hl = sum(1 for e in events if e == "HL")
        n_lh = sum(1 for e in events if e == "LH")
        n_ll = sum(1 for e in events if e == "LL")
        n_ev = max(len(events), 1)
        hh_count[t] = float(n_hh)
        hl_count[t] = float(n_hl)
        lh_count[t] = float(n_lh)
        ll_count[t] = float(n_ll)
        structure_bias[t] = (n_hh + n_hl - n_lh - n_ll) / float(n_ev)
        last_swing_dir[t] = last_dir
        bars_since_swing[t] = float(t - last_swing_idx) if last_swing_idx >= 0 else float(t)
        if np.isfinite(last_high_price) and np.isfinite(last_low_price):
            swing_amp_atr[t] = _clip_atr(abs(last_high_price - last_low_price) / atr_t)
        unconfirmed_ext_atr[t] = _clip_atr((e_price - close[t]) / atr_t)

        swing_lows = [p for _, p in lows if p <= close[t]]
        swing_highs = [p for _, p in highs if p >= close[t]]
        support = max(swing_lows) if swing_lows else float(donch_l[t])
        resistance = min(swing_highs) if swing_highs else float(donch_h[t])
        dist_to_support_atr[t] = _clip_atr((close[t] - support) / atr_t)
        dist_to_resistance_atr[t] = _clip_atr((resistance - close[t]) / atr_t)
        delta = _SR_DELTA_ATR * atr_t
        if low[t] <= support + delta and close[t] >= support - delta:
            support_touch[t] = 1.0
        if high[t] >= resistance - delta and close[t] <= resistance + delta:
            resistance_touch[t] = 1.0

        up_ext = (close[t] - donch_h[t]) / atr_t
        dn_ext = (donch_l[t] - close[t]) / atr_t
        if up_ext >= _BREAKOUT_THRESH:
            last_bo_idx = t
            last_bo_level = float(donch_h[t])
            last_bo_side = 1
        elif dn_ext >= _BREAKOUT_THRESH:
            last_bo_idx = t
            last_bo_level = float(donch_l[t])
            last_bo_side = -1
        if last_bo_idx >= 0:
            bars_since_breakout[t] = float(t - last_bo_idx)
            retest_dist_atr[t] = _clip_atr((close[t] - last_bo_level) / atr_t)
            if last_bo_side > 0 and close[t] < last_bo_level:
                failed_break[t] = 1.0
            elif last_bo_side < 0 and close[t] > last_bo_level:
                failed_break[t] = 1.0
        else:
            bars_since_breakout[t] = float(t)

        if len(highs) >= 2:
            (i1, p1), (i2, p2) = highs[-2], highs[-1]
            peak_diff_atr[t] = _clip_atr((p2 - p1) / atr_t)
            peak_sep_bars[t] = float(i2 - i1)
            between = [lp for li, lp in lows if i1 < li < i2]
            if between:
                neck = min(between)
                dist_neck_atr[t] = _clip_atr((close[t] - neck) / atr_t)
        if len(lows) >= 2:
            (j1, q1), (j2, q2) = lows[-2], lows[-1]
            trough_diff_atr[t] = _clip_atr((q2 - q1) / atr_t)
            trough_sep_bars[t] = float(j2 - j1)

        if len(highs) >= _CHANNEL_K:
            hx = np.array([idx for idx, _ in highs[-_CHANNEL_K:]], dtype=np.float64)
            hy = np.array([px for _, px in highs[-_CHANNEL_K:]], dtype=np.float64)
            hs = _ols_slope(hx, hy)
            high_slope_atr[t] = float(np.clip(hs / atr_t, -_SLOPE_CLIP, _SLOPE_CLIP))
        if len(lows) >= _CHANNEL_K:
            lx = np.array([idx for idx, _ in lows[-_CHANNEL_K:]], dtype=np.float64)
            ly = np.array([px for _, px in lows[-_CHANNEL_K:]], dtype=np.float64)
            ls = _ols_slope(lx, ly)
            low_slope_atr[t] = float(np.clip(ls / atr_t, -_SLOPE_CLIP, _SLOPE_CLIP))
        convergence[t] = float(
            np.clip(high_slope_atr[t] - low_slope_atr[t], -_SLOPE_CLIP, _SLOPE_CLIP)
        )
        if len(highs) >= 1 and len(lows) >= 1:
            h_line = highs[-1][1] + high_slope_atr[t] * atr_t * (t - highs[-1][0])
            l_line = lows[-1][1] + low_slope_atr[t] * atr_t * (t - lows[-1][0])
            width_now_atr[t] = _clip_atr((h_line - l_line) / atr_t)

    touch_sup = pd.Series(support_touch).rolling(_TOUCH_W, min_periods=1).sum()
    touch_res = pd.Series(resistance_touch).rolling(_TOUCH_W, min_periods=1).sum()

    out["hh_count"] = hh_count
    out["hl_count"] = hl_count
    out["lh_count"] = lh_count
    out["ll_count"] = ll_count
    out["structure_bias"] = structure_bias
    out["last_swing_dir"] = last_swing_dir
    out["bars_since_swing"] = bars_since_swing
    out["swing_amp_atr"] = swing_amp_atr
    out["unconfirmed_ext_atr"] = unconfirmed_ext_atr
    out["dist_to_support_atr"] = dist_to_support_atr
    out["dist_to_resistance_atr"] = dist_to_resistance_atr
    out["support_touch_count"] = touch_sup.to_numpy(dtype=np.float64)
    out["resistance_touch_count"] = touch_res.to_numpy(dtype=np.float64)
    out["bars_since_breakout"] = bars_since_breakout
    out["retest_dist_atr"] = retest_dist_atr
    out["failed_break"] = failed_break
    out["peak_diff_atr"] = peak_diff_atr
    out["trough_diff_atr"] = trough_diff_atr
    out["peak_sep_bars"] = peak_sep_bars
    out["trough_sep_bars"] = trough_sep_bars
    out["dist_neck_atr"] = dist_neck_atr
    out["high_slope_atr"] = high_slope_atr
    out["low_slope_atr"] = low_slope_atr
    out["convergence"] = convergence
    out["width_now_atr"] = width_now_atr
    out[CHART_PATTERN_COL] = _chart_pattern_ids(
        failed_break=failed_break,
        bars_since_breakout=bars_since_breakout,
        breakout_size_atr=(
            out["breakout_size_atr"].to_numpy(dtype=np.float64)
            if "breakout_size_atr" in out.columns
            else np.zeros(n)
        ),
        peak_diff_atr=peak_diff_atr,
        trough_diff_atr=trough_diff_atr,
        peak_sep_bars=peak_sep_bars,
        trough_sep_bars=trough_sep_bars,
        dist_neck_atr=dist_neck_atr,
        high_slope_atr=high_slope_atr,
        low_slope_atr=low_slope_atr,
        convergence=convergence,
        width_now_atr=width_now_atr,
        pole_disp_atr=(
            out["pole_disp_atr"].to_numpy(dtype=np.float64)
            if "pole_disp_atr" in out.columns
            else np.zeros(n)
        ),
        flag_width_atr=(
            out["flag_width_atr"].to_numpy(dtype=np.float64)
            if "flag_width_atr" in out.columns
            else np.zeros(n)
        ),
        flag_slope_atr=(
            out["flag_slope_atr"].to_numpy(dtype=np.float64)
            if "flag_slope_atr" in out.columns
            else np.zeros(n)
        ),
        structure_bias=structure_bias,
    )
    return out

def _chart_pattern_ids(
    *,
    failed_break: np.ndarray,
    bars_since_breakout: np.ndarray,
    breakout_size_atr: np.ndarray,
    peak_diff_atr: np.ndarray,
    trough_diff_atr: np.ndarray,
    peak_sep_bars: np.ndarray,
    trough_sep_bars: np.ndarray,
    dist_neck_atr: np.ndarray,
    high_slope_atr: np.ndarray,
    low_slope_atr: np.ndarray,
    convergence: np.ndarray,
    width_now_atr: np.ndarray,
    pole_disp_atr: np.ndarray,
    flag_width_atr: np.ndarray,
    flag_slope_atr: np.ndarray,
    structure_bias: np.ndarray,
) -> np.ndarray:
    """Causal discrete chart-pattern id at bar t (0=NONE ... 8=FAILED_BREAK).

    FAILED_BREAK is a rising-edge event inside ``_FAILED_BREAK_BARS`` of the
    last Donchian breakout. It does not overwrite FLAG/TRIANGLE/DOUBLE/CHANNEL
    and does not stay on for the whole post-breakout regime. BREAKOUT is the
    event bar only and also does not overwrite those named geometries.
    """
    _ = breakout_size_atr
    n = int(failed_break.shape[0])
    ids = np.zeros(n, dtype=np.int64)
    channel = (np.abs(high_slope_atr - low_slope_atr) < 0.15) & (width_now_atr > 0.6)
    triangle = (np.abs(convergence) >= 0.08) & (width_now_atr < 2.5)
    double_top = (
        (np.abs(peak_diff_atr) < 0.45)
        & (peak_sep_bars >= 4)
        & (dist_neck_atr > 0.15)
    )
    double_bottom = (
        (np.abs(trough_diff_atr) < 0.45)
        & (trough_sep_bars >= 4)
        & (dist_neck_atr < -0.15)
    )
    flag_bull = (
        (pole_disp_atr > 1.0)
        & (flag_width_atr < 2.0)
        & (flag_slope_atr < 0.0)
        & (structure_bias > 0.15)
    )
    flag_bear = (
        (pole_disp_atr < -1.0)
        & (flag_width_atr < 2.0)
        & (flag_slope_atr > 0.0)
        & (structure_bias < -0.15)
    )
    # Event bar only. abs(size vs Donchian high) is not a two-sided breakout
    # and was tagging most 5m bars as BREAKOUT.
    fresh_break = bars_since_breakout <= 0.5
    failed_now = (failed_break >= 0.5) & (
        bars_since_breakout <= float(_FAILED_BREAK_BARS)
    )
    failed_prev = np.empty_like(failed_now)
    failed_prev[0] = False
    if n > 1:
        failed_prev[1:] = failed_now[:-1]
    failed_pulse = failed_now & ~failed_prev
    ids = np.where(channel, 6, ids)
    ids = np.where(triangle, 3, ids)
    ids = np.where(flag_bull, 1, ids)
    ids = np.where(flag_bear, 2, ids)
    ids = np.where(double_bottom, 5, ids)
    ids = np.where(double_top, 4, ids)
    named = np.isin(ids, [1, 2, 3, 4, 5, 6])
    ids = np.where(fresh_break & ~named, 7, ids)
    ids = np.where(failed_pulse & ~named, 8, ids)
    return ids

def add_market_structure_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add causal native-TF structure, trend, S/R, breakout, and geometry columns.

    Args:
        df: OHLCV frame with atr (computed if missing).

    Returns:
        Same frame with ``NATIVE_STRUCTURE_COLS`` assigned and finite-filled.
    """
    out = _ensure_atr(df)
    out = _vectorized_trend_and_ma(out)
    out = _zigzag_structure_loop(out)
    for col in NATIVE_STRUCTURE_COLS:
        if col not in out.columns:
            out[col] = 0.0
        out[col] = out[col].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if CHART_PATTERN_COL not in out.columns:
        out[CHART_PATTERN_COL] = 0
    out[CHART_PATTERN_COL] = (
        pd.to_numeric(out[CHART_PATTERN_COL], errors="coerce").fillna(0).astype(np.int64)
    )
    return out

def _resample_closed_ohlcv(
    df: pd.DataFrame,
    htf_minutes: int,
    native_minutes: int = _HTF_NATIVE_MINUTES,
) -> pd.DataFrame:
    """Resample native bars to a higher TF, keeping only complete buckets."""
    expected = max(1, int(round(htf_minutes / native_minutes)))
    src = df[["time", "open", "high", "low", "close"]].copy()
    if "volume" in df.columns:
        src["volume"] = df["volume"]
    else:
        src["volume"] = 0.0
    src["time"] = pd.to_datetime(src["time"], utc=True)
    grouped = src.set_index("time")
    rule = f"{int(htf_minutes)}min"
    agg = grouped.resample(rule, label="right", closed="right").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample(rule, label="right", closed="right").count()
    complete = agg.loc[counts >= expected].dropna(subset=["open", "high", "low", "close"])
    if complete.empty:
        return complete
    complete = complete.reset_index()
    return complete

def add_htf_structure_features(df: pd.DataFrame) -> pd.DataFrame:
    """Merge closed 15m/30m/1h/2h structure onto a 5m frame (backward asof)."""
    out = df.copy()
    out["time"] = pd.to_datetime(out["time"], utc=True)
    out = out.sort_values("time").reset_index(drop=True)

    for tf_name in HTF_SOURCE_TFS:
        minutes = RESOLUTION_MINUTES[tf_name]
        htf_raw = _resample_closed_ohlcv(out, minutes, _HTF_NATIVE_MINUTES)
        prefixed = {f"htf_{tf_name}_{field}": 0.0 for field in HTF_FEATURE_FIELDS}
        if htf_raw.empty:
            for col, val in prefixed.items():
                out[col] = val
            continue
        htf_feat = add_market_structure_features(_ensure_atr(htf_raw))
        merge_cols = {"time": htf_feat["time"]}
        for field in HTF_FEATURE_FIELDS:
            src = _HTF_NATIVE_MAP[field]
            merge_cols[f"htf_{tf_name}_{field}"] = htf_feat[src].to_numpy(dtype=np.float64)
        htf_df = pd.DataFrame(merge_cols).sort_values("time")
        out = pd.merge_asof(
            out.sort_values("time"),
            htf_df,
            on="time",
            direction="backward",
        )

    for col in HTF_STRUCTURE_COLS:
        if col not in out.columns:
            out[col] = 0.0
        out[col] = out[col].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return out


## Feature engineering

In [ ]:
"""Causal per-TF feature engineering for BTCUSD transformers."""

from __future__ import annotations

from typing import Optional, Sequence

import numpy as np
import pandas as pd

WICK_NEGLIGIBLE = 0.05
WICK_BALANCE_MAX = 0.25
FLAT_ATR_MULT = 1e-4
_RATIO_COLS = ("body_ratio", "upper_wick_ratio", "lower_wick_ratio")
_STRUCTURE_EPS = 1e-9
_ATR_NORM_CLIP = 8.0

def assemble_raw_frame(
    df: pd.DataFrame,
    *,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """Merge OHLCV with funding/OI using the same path as live inference."""
    out = df.copy()
    if "timestamp" in out.columns and "time" not in out.columns:
        out["time"] = pd.to_datetime(out["timestamp"], utc=True)
    elif "time" in out.columns:
        out["time"] = pd.to_datetime(out["time"], utc=True)
    else:
        raise ValueError("OHLCV frame requires timestamp or time column")

    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"OHLCV frame missing columns: {missing}")

    out = out.sort_values("time").reset_index(drop=True)

    if funding_df is not None and not funding_df.empty:
        fund = funding_df.copy()
        if "timestamp" in fund.columns:
            fund["time"] = pd.to_datetime(fund["timestamp"], utc=True)
        rate_col = "funding_rate" if "funding_rate" in fund.columns else "close"
        fund = fund[["time", rate_col]].rename(columns={rate_col: "funding_rate"})
        out = pd.merge_asof(
            out.sort_values("time"),
            fund.sort_values("time"),
            on="time",
            direction="backward",
        )
    elif "funding_rate" not in out.columns:
        out["funding_rate"] = np.nan

    if oi_df is not None and not oi_df.empty:
        oi = oi_df.copy()
        if "timestamp" in oi.columns:
            oi["time"] = pd.to_datetime(oi["timestamp"], utc=True)
        oi_col = None
        for candidate in ("open_interest", "oi_contracts", "close"):
            if candidate in oi.columns:
                oi_col = candidate
                break
        if oi_col is not None:
            oi = oi[["time", oi_col]].rename(columns={oi_col: "open_interest"})
            out = pd.merge_asof(
                out.sort_values("time"),
                oi.sort_values("time"),
                on="time",
                direction="backward",
            )
    elif "open_interest" not in out.columns:
        out["open_interest"] = np.nan

    if "funding_rate" in out.columns:
        out["funding_rate"] = out["funding_rate"].ffill()
    if "open_interest" in out.columns:
        out["open_interest"] = out["open_interest"].ffill()

    return out

def prepare_raw_frame(
    df: pd.DataFrame,
    *,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """Alias for assemble_raw_frame (agent inference entry point)."""
    return assemble_raw_frame(df, funding_df=funding_df, oi_df=oi_df)

def classify_candle_shape(out: pd.DataFrame, *, sr_window: int) -> pd.Series:
    """Assign mutually exclusive single-bar candle class ids 0..12.

    Uses bar-i OHLC geometry plus rolling body-size quantiles up to bar i.
    First matching ``np.select`` condition wins.

    Args:
        out: Frame with open/high/low/close, atr, and wick/body ratio columns.
        sr_window: Rolling lookback for doji/marubozu quantiles.

    Returns:
        int64 Series of class ids aligned with ``out.index``.
    """
    rng_raw = (out["high"] - out["low"]).astype(float)
    atr = out["atr"] if "atr" in out.columns else pd.Series(0.0, index=out.index)
    is_flat = (rng_raw.fillna(0) <= 0) | (
        rng_raw.fillna(0) < atr.fillna(0) * FLAT_ATR_MULT
    )

    body_abs = out["body_ratio"].abs()
    upper = out["upper_wick_ratio"]
    lower = out["lower_wick_ratio"]
    doji_thresh = body_abs.rolling(sr_window, min_periods=20).quantile(0.15)
    marubozu_thresh = body_abs.rolling(sr_window, min_periods=20).quantile(0.85)
    body_median = body_abs.rolling(sr_window, min_periods=20).median()

    wick_sum = upper + lower
    wick_imbalance = (upper - lower).abs() / (wick_sum + 1e-9)
    cond_wicks_balanced = (wick_sum > 2 * WICK_NEGLIGIBLE) & (
        wick_imbalance < WICK_BALANCE_MAX
    )

    cond_doji = ~is_flat & (body_abs < doji_thresh)
    cond_dragonfly = (
        cond_doji
        & (upper < WICK_NEGLIGIBLE)
        & (lower > 2 * WICK_NEGLIGIBLE)
    )
    cond_gravestone = (
        cond_doji
        & (lower < WICK_NEGLIGIBLE)
        & (upper > 2 * WICK_NEGLIGIBLE)
    )
    cond_doji_standard = cond_doji
    cond_marubozu_bull = (
        ~is_flat
        & ~cond_doji
        & (out["body_ratio"] > marubozu_thresh)
        & (upper < WICK_NEGLIGIBLE)
        & (lower < WICK_NEGLIGIBLE)
    )
    cond_marubozu_bear = (
        ~is_flat
        & ~cond_doji
        & (-out["body_ratio"] > marubozu_thresh)
        & (upper < WICK_NEGLIGIBLE)
        & (lower < WICK_NEGLIGIBLE)
    )
    cond_hammer = (
        ~is_flat
        & ~cond_doji
        & (lower > 2 * body_abs)
        & (upper < body_abs)
    )
    cond_inv_hammer = (
        ~is_flat
        & ~cond_doji
        & (upper > 2 * body_abs)
        & (lower < body_abs)
    )
    cond_spinning = (
        ~is_flat
        & ~cond_doji
        & (body_abs < body_median)
        & cond_wicks_balanced
    )
    cond_belt_bull = (
        ~is_flat
        & ~cond_doji
        & (out["close"] > out["open"])
        & (lower < WICK_NEGLIGIBLE)
        & (upper > WICK_NEGLIGIBLE)
    )
    cond_belt_bear = (
        ~is_flat
        & ~cond_doji
        & (out["close"] < out["open"])
        & (upper < WICK_NEGLIGIBLE)
        & (lower > WICK_NEGLIGIBLE)
    )
    cond_standard_bull = ~is_flat & (out["close"] > out["open"])
    cond_standard_bear = ~is_flat & (out["close"] < out["open"])

    conditions = [
        is_flat,
        cond_dragonfly,
        cond_gravestone,
        cond_doji_standard,
        cond_marubozu_bull,
        cond_marubozu_bear,
        cond_hammer,
        cond_inv_hammer,
        cond_spinning,
        cond_belt_bull,
        cond_belt_bear,
        cond_standard_bull,
        cond_standard_bear,
    ]
    choices = list(range(CANDLE_CLASS_CARDINALITY))
    ids = np.select(conditions, choices, default=3).astype(np.int64)
    return pd.Series(ids, index=out.index, dtype="int64")

def summarize_candle_class_distribution(
    feat_df: pd.DataFrame,
    *,
    high_frac: float = 0.40,
    low_frac: float = 0.001,
) -> pd.Series:
    """Return class fractions and print warnings for extreme imbalance.

    Args:
        feat_df: Feature frame containing ``candle_class_id``.
        high_frac: Warn if any class exceeds this share of bars.
        low_frac: Warn if any present class is below this share.

    Returns:
        Normalized value counts indexed by class id.
    """
    counts = feat_df[CANDLE_CLASS_COL].value_counts(normalize=True).sort_index()
    for cid, frac in counts.items():
        if float(frac) > high_frac or float(frac) < low_frac:
            print(
                f"WARNING: candle class {int(cid)} fraction {float(frac):.4f} "
                f"(thresholds {low_frac:.4f} / {high_frac:.2f})"
            )
    return counts

def add_candle_structure_features(out: pd.DataFrame) -> pd.DataFrame:
    """Add causal bar-geometry and bar-to-bar relation features.

    All columns use OHLCV and ATR at or before bar t. Shift-based values on
    the first bar fill to 0.

    Args:
        out: Frame with open/high/low/close and atr.

    Returns:
        The same frame with seven structure columns assigned.
    """
    rng_raw = (out["high"] - out["low"]).astype(float)
    atr = out["atr"] if "atr" in out.columns else pd.Series(0.0, index=out.index)
    is_flat = rng_raw.fillna(0) <= 0

    close_loc = (out["close"] - out["low"]) / (rng_raw + _STRUCTURE_EPS)
    out["close_loc"] = close_loc.where(~is_flat, 0.5).fillna(0.5)

    out["range_atr"] = (rng_raw / (atr + _STRUCTURE_EPS)).clip(
        -_ATR_NORM_CLIP, _ATR_NORM_CLIP
    )
    out["body_atr"] = (
        (out["close"] - out["open"]).abs() / (atr + _STRUCTURE_EPS)
    ).clip(-_ATR_NORM_CLIP, _ATR_NORM_CLIP)
    out["gap_atr"] = (
        (out["open"] - out["close"].shift(1)) / (atr + _STRUCTURE_EPS)
    ).clip(-_ATR_NORM_CLIP, _ATR_NORM_CLIP)

    prev_high = out["high"].shift(1)
    prev_low = out["low"].shift(1)
    out["inside_bar"] = (
        (out["high"] <= prev_high) & (out["low"] >= prev_low)
    ).astype(np.float32)
    out["outside_bar"] = (
        (out["high"] >= prev_high) & (out["low"] <= prev_low)
    ).astype(np.float32)

    prior_body = out["close"].shift(1) - out["open"].shift(1)
    cur_body = out["close"] - out["open"]
    opposite = np.sign(cur_body) != np.sign(prior_body)
    prior_nonzero = prior_body.abs() > _STRUCTURE_EPS
    penetration = (out["close"] - out["open"].shift(1)) / (
        prior_body.abs() + _STRUCTURE_EPS
    )
    out["engulf_score"] = (
        penetration.clip(-2.0, 2.0)
        * np.sign(cur_body)
        * opposite.astype(np.float64)
        * prior_nonzero.astype(np.float64)
    )

    shift_fill_cols = ("gap_atr", "inside_bar", "outside_bar", "engulf_score")
    for col in shift_fill_cols:
        out[col] = out[col].fillna(0.0)
    return out

def add_features(
    df: pd.DataFrame,
    *,
    resolution_minutes: int = 15,
    atr_period: int = 14,
    rsi_period: int = 14,
    adx_period: int = 14,
    ema_period: int = 50,
    macd_fast: int = 12,
    macd_slow: int = 26,
    macd_signal: int = 9,
) -> pd.DataFrame:
    """Compute causal features on a native TF grid."""
    out = df.copy()
    rv_short = scale_period(16, resolution_minutes)
    rv_long = scale_period(96, resolution_minutes)
    sr_window = scale_period(96, resolution_minutes)
    obv_window = scale_period(96, resolution_minutes)
    vol_window = scale_period(96, resolution_minutes)
    ret2_bars = max(1, scale_period(2, resolution_minutes))

    atr_period = scale_period(atr_period, resolution_minutes)
    rsi_period = scale_period(rsi_period, resolution_minutes)
    adx_period = scale_period(adx_period, resolution_minutes)
    ema_period = scale_period(ema_period, resolution_minutes)
    macd_fast = scale_period(macd_fast, resolution_minutes)
    macd_slow = scale_period(macd_slow, resolution_minutes)
    macd_signal = scale_period(macd_signal, resolution_minutes)

    out["ret_1"] = np.log(out["close"] / out["close"].shift(1))

    prev_close = out["close"].shift(1)
    tr = pd.concat(
        [
            out["high"] - out["low"],
            (out["high"] - prev_close).abs(),
            (out["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    out["atr"] = tr.rolling(atr_period).mean()

    out["rv_16"] = out["ret_1"].rolling(rv_short).std()
    out["rv_96"] = out["ret_1"].rolling(rv_long).std()

    out["ema50"] = out["close"].ewm(span=ema_period, adjust=False).mean()
    out["ema50_dist_pct"] = (out["close"] - out["ema50"]) / out["ema50"]

    ema_fast_s = out["close"].ewm(span=macd_fast, adjust=False).mean()
    ema_slow_s = out["close"].ewm(span=macd_slow, adjust=False).mean()
    macd_line = ema_fast_s - ema_slow_s
    macd_signal_line = macd_line.ewm(span=macd_signal, adjust=False).mean()
    out["macd_hist"] = macd_line - macd_signal_line

    delta = out["close"].diff()
    gain = delta.clip(lower=0).rolling(rsi_period).mean()
    loss = (-delta.clip(upper=0)).rolling(rsi_period).mean()
    rs = gain / (loss + 1e-9)
    out["rsi_14"] = 100 - (100 / (1 + rs))

    up_move = out["high"].diff()
    down_move = -out["low"].diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    atr_for_di = tr.rolling(adx_period).mean()
    plus_di = 100 * pd.Series(plus_dm, index=out.index).rolling(adx_period).mean() / (
        atr_for_di + 1e-9
    )
    minus_di = 100 * pd.Series(minus_dm, index=out.index).rolling(adx_period).mean() / (
        atr_for_di + 1e-9
    )
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-9)
    out["adx_14"] = dx.rolling(adx_period).mean()

    obv_raw = (np.sign(out["close"].diff()) * out["volume"]).fillna(0).cumsum()
    out["obv_z"] = (obv_raw - obv_raw.rolling(obv_window).mean()) / (
        obv_raw.rolling(obv_window).std() + 1e-9
    )

    out["vol_z"] = (out["volume"] - out["volume"].rolling(vol_window).mean()) / (
        out["volume"].rolling(vol_window).std() + 1e-9
    )

    rng_raw = (out["high"] - out["low"]).astype(float)
    is_flat = (rng_raw.fillna(0) <= 0) | (
        rng_raw.fillna(0) < out["atr"].fillna(0) * FLAT_ATR_MULT
    )
    rng = rng_raw.replace(0, np.nan)
    out["body_ratio"] = (out["close"] - out["open"]) / rng
    out["upper_wick_ratio"] = (
        out["high"] - out[["open", "close"]].max(axis=1)
    ) / rng
    out["lower_wick_ratio"] = (
        out[["open", "close"]].min(axis=1) - out["low"]
    ) / rng
    out.loc[is_flat, list(_RATIO_COLS)] = 0.0
    out[list(_RATIO_COLS)] = out[list(_RATIO_COLS)].fillna(0.0)

    out["hour"] = out["time"].dt.hour
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["dow"] = out["time"].dt.dayofweek
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7)

    rolling_high = out["high"].rolling(sr_window).max()
    rolling_low = out["low"].rolling(sr_window).min()
    out["dist_to_resistance_pct"] = (rolling_high - out["close"]) / out["close"]
    out["dist_to_support_pct"] = (out["close"] - rolling_low) / out["close"]

    if "funding_rate" not in out.columns:
        out["funding_rate"] = np.nan

    if "open_interest" in out.columns:
        oi_z_window = scale_period(96, resolution_minutes)
        out["oi_z"] = (out["open_interest"] - out["open_interest"].rolling(oi_z_window).mean()) / (
            out["open_interest"].rolling(oi_z_window).std() + 1e-9
        )
    else:
        out["oi_z"] = np.nan

    ret_2 = out["close"].pct_change(ret2_bars)
    fund_rate = out["funding_rate"].fillna(0.0)
    funding_deriv = compute_funding_derivatives(
        fund_rate, ret_2, resolution_minutes=resolution_minutes
    )
    for col in funding_deriv.columns:
        out[col] = funding_deriv[col].values

    oi_deriv = compute_oi_derivatives(out, resolution_minutes=resolution_minutes)
    out["oi_change_2"] = oi_deriv["oi_change_2"]
    out["oi_delta_z"] = oi_deriv["oi_delta_z"]
    out["oi_price_divergence"] = oi_deriv["oi_price_divergence"]
    out["oi_acceleration"] = oi_deriv["oi_acceleration"]
    out["funding_x_oi"] = out["funding_zscore"] * oi_deriv["oi_zscore"]

    out = add_candle_structure_features(out)
    out[CANDLE_CLASS_COL] = classify_candle_shape(out, sr_window=sr_window)
    out = add_market_structure_features(out)
    if int(resolution_minutes) == 5:
        out = add_htf_structure_features(out)

    return out

def build_feature_matrix(
    df: pd.DataFrame,
    *,
    resolution_minutes: int = 15,
    atr_period: int = 14,
    dropna: bool = True,
) -> pd.DataFrame:
    """Return feature columns ready for windowing."""
    feat = add_features(df, resolution_minutes=resolution_minutes, atr_period=atr_period)
    if dropna:
        feat = feat.dropna().reset_index(drop=True)
    return feat

def latest_closed_feature_row(feat_df: pd.DataFrame) -> pd.Series:
    """Closed-bar row used for diagnostics (second-to-last after dropna)."""
    if len(feat_df) < 2:
        raise ValueError("Need at least 2 feature rows for closed-bar semantics")
    return feat_df.iloc[-2]

def validate_feature_columns(
    feat_df: pd.DataFrame,
    *,
    require_finite_closed_bar: bool = False,
    feature_cols: Optional[Sequence[str]] = None,
) -> None:
    cols = tuple(feature_cols) if feature_cols is not None else FEATURE_COLS
    missing = [c for c in cols if c not in feat_df.columns]
    if missing:
        raise ValueError(f"Feature matrix missing columns: {missing}")
    if CANDLE_CLASS_COL not in feat_df.columns:
        raise ValueError(f"Feature matrix missing {CANDLE_CLASS_COL}")
    ids = feat_df[CANDLE_CLASS_COL]
    if ((ids < 0) | (ids > CANDLE_CLASS_CARDINALITY - 1)).any():
        raise ValueError(
            f"{CANDLE_CLASS_COL} out of range [0, {CANDLE_CLASS_CARDINALITY - 1}]"
        )
    if require_finite_closed_bar and len(feat_df) >= 2:
        closed = latest_closed_feature_row(feat_df)
        for col in cols:
            val = closed[col]
            if not np.isfinite(float(val)):
                raise ValueError(f"Non-finite closed-bar value for {col}: {val}")
        cid = int(closed[CANDLE_CLASS_COL])
        if cid < 0 or cid > CANDLE_CLASS_CARDINALITY - 1:
            raise ValueError(f"Closed-bar {CANDLE_CLASS_COL} out of range: {cid}")


## Labels / targets

In [ ]:
"""Forward-looking market labels for per-TF transformer training."""

from __future__ import annotations

from typing import Dict, Sequence

import numpy as np
import pandas as pd

_EPS = 1e-9
_STRUCTURE_OUTCOME_RANGE = 0
_STRUCTURE_OUTCOME_CONT_LONG = 1
_STRUCTURE_OUTCOME_CONT_SHORT = 2
_STRUCTURE_OUTCOME_BREAKOUT = 3
_STRUCTURE_OUTCOME_FAILED_BREAK = 4
_STRUCTURE_OUTCOME_REVERSAL = 5

def _sign_nonzero(value: float) -> float:
    if not np.isfinite(value) or abs(value) <= _EPS:
        return 0.0
    return 1.0 if value > 0.0 else -1.0

def _structure_outcome_id(
    *,
    bias_now: float,
    bias_end: float,
    mfe: float,
    mae: float,
    fwd_failed: np.ndarray,
    fwd_bars_since_breakout: np.ndarray,
    event_scope: str = "window",
) -> int:
    """Priority: failed break, breakout, reversal, continuation, else range.

    ``event_scope="window"`` (v6) ORs event flags over the whole path.
    ``event_scope="terminal"`` (v8) uses only the last bar so 1h/2h windows
    do not collapse to BREAKOUT whenever a Donchian print occurs anywhere.
    """
    if event_scope == "terminal":
        failed_hit = bool(fwd_failed.size) and float(fwd_failed[-1]) >= 0.5
        breakout_hit = (
            bool(fwd_bars_since_breakout.size)
            and float(fwd_bars_since_breakout[-1]) <= 0.5
        )
    else:
        failed_hit = bool(fwd_failed.size) and bool(np.any(fwd_failed >= 0.5))
        breakout_hit = (
            bool(fwd_bars_since_breakout.size)
            and bool(np.any(fwd_bars_since_breakout <= 0.5))
        )
    if failed_hit:
        return _STRUCTURE_OUTCOME_FAILED_BREAK
    if breakout_hit:
        return _STRUCTURE_OUTCOME_BREAKOUT
    s0 = _sign_nonzero(bias_now)
    s1 = _sign_nonzero(bias_end)
    if s0 != 0.0 and s1 != 0.0 and s0 != s1:
        return _STRUCTURE_OUTCOME_REVERSAL
    if bias_end > 0.0 and float(mfe) > float(mae):
        return _STRUCTURE_OUTCOME_CONT_LONG
    if bias_end < 0.0 and float(mae) > float(mfe):
        return _STRUCTURE_OUTCOME_CONT_SHORT
    return _STRUCTURE_OUTCOME_RANGE

def _realized_path_vol(fwd_rets: np.ndarray) -> float:
    """RMS log-return. For k=1 this is |r| instead of a degenerate std of one sample."""
    if fwd_rets.size == 0:
        return 0.0
    return float(np.sqrt(np.mean(np.square(fwd_rets))))

def compute_market_labels(
    df: pd.DataFrame,
    *,
    path_label_horizon_bars: int = PATH_LABEL_HORIZON_BARS,
    mae_floor_atr_mult: float,
) -> pd.DataFrame:
    """Compute path/risk and forward-pattern labels on the native TF grid."""
    path_horizon = int(path_label_horizon_bars)
    max_horizon = max_label_horizon_bars(path_horizon)

    out_df = df.copy()
    n = len(out_df)
    close = out_df["close"].values
    high = out_df["high"].values
    low = out_df["low"].values
    open_px = out_df["open"].values
    atr = out_df["atr"].values
    volume = out_df["volume"].values
    has_oi = "open_interest" in out_df.columns
    oi = out_df["open_interest"].values if has_oi else np.full(n, np.nan)
    bias = (
        out_df["structure_bias"].to_numpy(dtype=np.float64)
        if "structure_bias" in out_df.columns
        else np.zeros(n, dtype=np.float64)
    )
    failed = (
        out_df["failed_break"].to_numpy(dtype=np.float64)
        if "failed_break" in out_df.columns
        else np.zeros(n, dtype=np.float64)
    )
    bars_bo = (
        out_df["bars_since_breakout"].to_numpy(dtype=np.float64)
        if "bars_since_breakout" in out_df.columns
        else np.full(n, np.inf)
    )
    candle_ids = (
        out_df[CANDLE_CLASS_COL].to_numpy(dtype=np.int64)
        if CANDLE_CLASS_COL in out_df.columns
        else np.zeros(n, dtype=np.int64)
    )

    path_cols = list(CONTINUOUS_LABEL_COLS)
    out: dict[str, np.ndarray] = {c: np.full(n, np.nan) for c in path_cols}
    structure_ids = np.full(n, np.nan)
    future_candle = np.full(n, np.nan)

    for i in range(n - max_horizon):
        entry = close[i]
        atr_i = float(atr[i]) if np.isfinite(atr[i]) else _EPS
        atr_i = max(atr_i, _EPS)

        fwd_close = close[i + 1 : i + path_horizon + 1]
        fwd_high = high[i + 1 : i + path_horizon + 1]
        fwd_low = low[i + 1 : i + path_horizon + 1]

        fwd_rets = np.log(fwd_close / np.concatenate(([entry], fwd_close[:-1])))
        out["future_volatility"][i] = fwd_rets.std()

        favorable = (fwd_high - entry) / entry
        adverse = (entry - fwd_low) / entry
        mfe_idx = int(np.argmax(favorable))
        out["mfe"][i] = favorable[mfe_idx]
        out["mae"][i] = adverse[int(np.argmax(adverse))]

        if out["mfe"][i] >= mae_floor_atr_mult * atr[i] / entry:
            out["drawdown_before_mfe"][i] = (
                adverse[: mfe_idx + 1].max() if mfe_idx > 0 else 0.0
            )

        out["trend_strength"][i] = abs(fwd_close[-1] - entry) / (atr_i)

        if has_oi and not np.isnan(oi[i]) and oi[i] != 0:
            out["future_oi_change_pct"][i] = (oi[i + path_horizon] - oi[i]) / (
                abs(oi[i]) + _EPS
            )
        past_start = max(0, i - path_horizon)
        past_vol_mean = volume[past_start : i + 1].mean()
        out["future_volume_change_pct"][i] = (
            volume[i + 1 : i + path_horizon + 1].mean() - past_vol_mean
        ) / (past_vol_mean + _EPS)

        move_atr = (close[i + 1] - close[i]) / atr_i
        body = float(close[i] - open_px[i])
        cid = int(candle_ids[i])
        if cid in CANDLE_FAMILY_DOJI or abs(body) <= _EPS:
            out["candle_follow_through_atr"][i] = abs(move_atr)
        else:
            out["candle_follow_through_atr"][i] = float(np.sign(body)) * move_atr

        end = i + path_horizon
        out["structure_delta"][i] = float(bias[end] - bias[i])
        future_candle[i] = float(candle_ids[i + 1])
        structure_ids[i] = float(
            _structure_outcome_id(
                bias_now=float(bias[i]),
                bias_end=float(bias[end]),
                mfe=float(out["mfe"][i]),
                mae=float(out["mae"][i]),
                fwd_failed=failed[i + 1 : i + path_horizon + 1],
                fwd_bars_since_breakout=bars_bo[i + 1 : i + path_horizon + 1],
            )
        )

    for col, values in out.items():
        out_df[col] = values
    out_df[STRUCTURE_OUTCOME_COL] = structure_ids
    out_df[FUTURE_CANDLE_COL] = future_candle
    return out_df

def trim_label_tail(
    df: pd.DataFrame,
    *,
    path_label_horizon_bars: int = PATH_LABEL_HORIZON_BARS,
) -> pd.DataFrame:
    """Drop rows without complete forward labels."""
    max_horizon = max_label_horizon_bars(path_label_horizon_bars)
    return df.iloc[: -(max_horizon + 1)].reset_index(drop=True)

def active_training_label_cols() -> tuple[str, ...]:
    return CONTINUOUS_LABEL_COLS

def label_nan_summary(df: pd.DataFrame) -> Dict[str, float]:
    """Per-label NaN rates for notebook diagnostics."""
    cols = [c for c in CONTINUOUS_LABEL_COLS if c in df.columns]
    if not cols:
        return {}
    return df[cols].isna().mean().round(4).to_dict()

def _volume_state_id(vol_z: float, breakout_vol_ratio: float) -> int:
    if np.isfinite(vol_z) and vol_z >= VOL_Z_EXPANSION:
        return 2
    if np.isfinite(breakout_vol_ratio) and breakout_vol_ratio >= BREAKOUT_VOL_CONFIRM:
        return 2
    if np.isfinite(vol_z) and vol_z <= VOL_Z_DRY:
        return 0
    return 1

def _volume_confirms(pattern_id: int, volume_state: int) -> int:
    pid = int(pattern_id)
    vs = int(volume_state)
    if pid == 0:
        return 0
    if pid in (1, 2, 7, 4, 5, 8) and vs != 2:
        return 0
    if pid == 3 and vs == 2:
        return 0
    return 1

def _wick_class(upper: float, lower: float) -> int:
    u = float(upper) if np.isfinite(upper) else 0.0
    lo = float(lower) if np.isfinite(lower) else 0.0
    if u > 0.30 and lo > 0.30:
        return 3
    if u > 2.0 * max(lo, 1e-6) and u > 0.30:
        return 1
    if lo > 2.0 * max(u, 1e-6) and lo > 0.30:
        return 2
    return 0

def _horizon_direction(move: float, atr: float) -> int:
    """5-class ATR-normalized direction: STRONG_DOWN..STRONG_UP."""
    ratio = float(move) / max(float(atr), 1e-9)
    if ratio > HORIZON_DIR_ATR_STRONG:
        return 4
    if ratio >= HORIZON_DIR_ATR_WEAK:
        return 3
    if ratio < -HORIZON_DIR_ATR_STRONG:
        return 0
    if ratio <= -HORIZON_DIR_ATR_WEAK:
        return 1
    return 2

def compute_horizon_behavior_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Label wall-clock path behavior at 5m/10m/15m/30m/1h/2h on the 5m grid.

    Features at bar t stay causal. Targets at t use bars t+1 ... t+k only.
    """
    out_df = df.copy()
    n = len(out_df)
    max_k = int(MAX_V8_HORIZON_BARS)
    close = out_df["close"].to_numpy(dtype=np.float64)
    high = out_df["high"].to_numpy(dtype=np.float64)
    low = out_df["low"].to_numpy(dtype=np.float64)
    atr = (
        out_df["atr"].to_numpy(dtype=np.float64)
        if "atr" in out_df.columns
        else np.full(n, np.nan)
    )
    bias = (
        out_df["structure_bias"].to_numpy(dtype=np.float64)
        if "structure_bias" in out_df.columns
        else np.zeros(n, dtype=np.float64)
    )
    failed = (
        out_df["failed_break"].to_numpy(dtype=np.float64)
        if "failed_break" in out_df.columns
        else np.zeros(n, dtype=np.float64)
    )
    bars_bo = (
        out_df["bars_since_breakout"].to_numpy(dtype=np.float64)
        if "bars_since_breakout" in out_df.columns
        else np.full(n, np.inf)
    )
    vol_z = (
        out_df["vol_z"].to_numpy(dtype=np.float64)
        if "vol_z" in out_df.columns
        else np.zeros(n)
    )
    bo_vol = (
        out_df["breakout_vol_ratio"].to_numpy(dtype=np.float64)
        if "breakout_vol_ratio" in out_df.columns
        else np.ones(n)
    )

    dir_out = {col: np.full(n, np.nan) for col in HORIZON_DIR_COLS}
    struct_out = {col: np.full(n, np.nan) for col in HORIZON_STRUCTURE_COLS}
    cont_out = {col: np.full(n, np.nan) for col in V8_CONTINUOUS_LABEL_COLS}
    vol_state = np.full(n, np.nan)

    for i in range(n - max_k):
        entry = float(close[i])
        atr_i = float(atr[i]) if np.isfinite(atr[i]) else _EPS
        atr_i = max(atr_i, _EPS)
        vol_state[i] = float(_volume_state_id(float(vol_z[i]), float(bo_vol[i])))
        for key, k in HORIZON_SPECS:
            kk = int(k)
            fwd_close = close[i + 1 : i + kk + 1]
            fwd_high = high[i + 1 : i + kk + 1]
            fwd_low = low[i + 1 : i + kk + 1]
            if fwd_close.size < kk:
                continue
            move = float(close[i + kk] - close[i])
            dir_out[f"{key}_dir"][i] = float(_horizon_direction(move, atr_i))
            log_base = np.concatenate(([entry], fwd_close[:-1]))
            fwd_rets = np.log(
                np.maximum(fwd_close, _EPS) / np.maximum(log_base, _EPS)
            )
            cont_out[f"{key}_vol"][i] = _realized_path_vol(fwd_rets)
            scale = max(abs(entry), _EPS)
            favorable = (fwd_high - entry) / scale
            adverse = (entry - fwd_low) / scale
            cont_out[f"{key}_mfe"][i] = float(favorable.max())
            cont_out[f"{key}_mae"][i] = float(adverse.max())
            cont_out[f"{key}_trend_strength"][i] = abs(
                float(fwd_close[-1] - entry)
            ) / atr_i
            end = i + kk
            struct_out[f"{key}_structure"][i] = float(
                _structure_outcome_id(
                    bias_now=float(bias[i]),
                    bias_end=float(bias[end]),
                    mfe=float(cont_out[f"{key}_mfe"][i]),
                    mae=float(cont_out[f"{key}_mae"][i]),
                    fwd_failed=failed[i + 1 : i + kk + 1],
                    fwd_bars_since_breakout=bars_bo[i + 1 : i + kk + 1],
                    event_scope="terminal",
                )
            )

    for col, values in dir_out.items():
        out_df[col] = values
    for col, values in struct_out.items():
        out_df[col] = values
    for col, values in cont_out.items():
        out_df[col] = values
    out_df[VOLUME_STATE_COL] = vol_state
    return out_df

def trim_v8_label_tail(df: pd.DataFrame) -> pd.DataFrame:
    """Drop rows without a complete longest-horizon (2h / 24-bar) label."""
    max_h = int(MAX_V8_HORIZON_BARS)
    if len(df) <= max_h + 1:
        return df.iloc[0:0].reset_index(drop=True)
    return df.iloc[: -(max_h + 1)].reset_index(drop=True)

def compute_next_candle_structure_labels(
    df: pd.DataFrame,
    *,
    path_label_horizon_bars: int = PATH_LABEL_HORIZON_BARS,
    horizon_bars: Sequence[int] = HORIZON_BARS_5M,
    gate_chart_volume: bool = True,
) -> pd.DataFrame:
    """Label t+1 candle structure, 5m horizon directions, and volume/chart gates.

    Target anatomy is computed from bar t+1 OHLC only. Bin edges for body/range
    use a causal rolling window through bar t (no future leakage).
    """
    out_df = df.copy()
    n = len(out_df)
    max_h = max(int(path_label_horizon_bars), max(int(k) for k in horizon_bars))
    close = out_df["close"].to_numpy(dtype=np.float64)
    open_px = out_df["open"].to_numpy(dtype=np.float64)
    high = out_df["high"].to_numpy(dtype=np.float64)
    low = out_df["low"].to_numpy(dtype=np.float64)
    atr = (
        out_df["atr"].to_numpy(dtype=np.float64)
        if "atr" in out_df.columns
        else np.full(n, np.nan)
    )
    body_ratio = (
        out_df["body_ratio"].to_numpy(dtype=np.float64)
        if "body_ratio" in out_df.columns
        else (close - open_px) / np.maximum(high - low, 1e-9)
    )
    upper = (
        out_df["upper_wick_ratio"].to_numpy(dtype=np.float64)
        if "upper_wick_ratio" in out_df.columns
        else np.zeros(n)
    )
    lower = (
        out_df["lower_wick_ratio"].to_numpy(dtype=np.float64)
        if "lower_wick_ratio" in out_df.columns
        else np.zeros(n)
    )
    range_atr = (
        out_df["range_atr"].to_numpy(dtype=np.float64)
        if "range_atr" in out_df.columns
        else (high - low) / np.maximum(atr, 1e-9)
    )
    vol_z = (
        out_df["vol_z"].to_numpy(dtype=np.float64)
        if "vol_z" in out_df.columns
        else np.zeros(n)
    )
    bo_vol = (
        out_df["breakout_vol_ratio"].to_numpy(dtype=np.float64)
        if "breakout_vol_ratio" in out_df.columns
        else np.ones(n)
    )
    pattern = (
        out_df[CHART_PATTERN_COL].to_numpy(dtype=np.int64)
        if CHART_PATTERN_COL in out_df.columns
        else np.zeros(n, dtype=np.int64)
    )
    candle_ids = (
        out_df[CANDLE_CLASS_COL].to_numpy(dtype=np.int64)
        if CANDLE_CLASS_COL in out_df.columns
        else np.zeros(n, dtype=np.int64)
    )

    abs_body = np.abs(body_ratio)
    q33 = (
        pd.Series(abs_body)
        .shift(1)
        .rolling(64, min_periods=20)
        .quantile(0.33)
        .to_numpy()
    )
    q66 = (
        pd.Series(abs_body)
        .shift(1)
        .rolling(64, min_periods=20)
        .quantile(0.66)
        .to_numpy()
    )
    rng_med = (
        pd.Series(range_atr).shift(1).rolling(64, min_periods=20).median().to_numpy()
    )

    nxt_dir = np.full(n, np.nan)
    nxt_body = np.full(n, np.nan)
    nxt_wick = np.full(n, np.nan)
    nxt_range = np.full(n, np.nan)
    vol_state = np.full(n, np.nan)
    vol_ok = np.zeros(n, dtype=np.float64)
    pat_active = np.zeros(n, dtype=np.float64)
    weight = np.zeros(n, dtype=np.float64)
    horizon_out = {col: np.full(n, np.nan) for col in HORIZON_DIR_COLS}

    for i in range(n - max_h):
        j = i + 1
        body_j = close[j] - open_px[j]
        atr_i = float(atr[i]) if np.isfinite(atr[i]) else 1e-9
        atr_i = max(atr_i, 1e-9)
        if abs(body_j) < 0.15 * atr_i:
            nxt_dir[i] = 1.0
        elif body_j > 0:
            nxt_dir[i] = 2.0
        else:
            nxt_dir[i] = 0.0

        ab = abs(float(body_ratio[j]))
        lo_b = float(q33[j]) if np.isfinite(q33[j]) else 0.30
        hi_b = float(q66[j]) if np.isfinite(q66[j]) else 0.70
        if ab <= lo_b:
            nxt_body[i] = 0.0
        elif ab >= hi_b:
            nxt_body[i] = 2.0
        else:
            nxt_body[i] = 1.0

        nxt_wick[i] = float(_wick_class(float(upper[j]), float(lower[j])))
        med = float(rng_med[j]) if np.isfinite(rng_med[j]) else 1.0
        ra = float(range_atr[j]) if np.isfinite(range_atr[j]) else med
        if ra < 0.70 * med:
            nxt_range[i] = 0.0
        elif ra > 1.40 * med:
            nxt_range[i] = 2.0
        else:
            nxt_range[i] = 1.0

        vs = _volume_state_id(float(vol_z[i]), float(bo_vol[i]))
        vol_state[i] = float(vs)
        pid = int(pattern[i])
        pat_active[i] = 0.0 if pid == 0 else 1.0
        vol_ok[i] = float(_volume_confirms(pid, vs))
        if gate_chart_volume:
            weight[i] = float(pat_active[i] >= 0.5 and vol_ok[i] >= 0.5)
        else:
            weight[i] = 1.0

        for col, k in zip(HORIZON_DIR_COLS, horizon_bars):
            kk = int(k)
            if i + kk >= n:
                continue
            move = close[i + kk] - close[i]
            horizon_out[col][i] = float(_horizon_direction(move, atr_i))

    out_df[NEXT_DIRECTION_COL] = nxt_dir
    out_df[NEXT_BODY_COL] = nxt_body
    out_df[NEXT_WICK_COL] = nxt_wick
    out_df[NEXT_RANGE_COL] = nxt_range
    out_df[VOLUME_STATE_COL] = vol_state
    out_df[VOLUME_CONFIRMS_COL] = vol_ok
    out_df[PATTERN_ACTIVE_COL] = pat_active
    out_df[SAMPLE_WEIGHT_COL] = weight
    for col, values in horizon_out.items():
        out_df[col] = values
    expected = np.array(
        [expected_direction_from_chart_pattern(int(p)) for p in pattern],
        dtype=np.float64,
    )
    agrees = (nxt_dir == expected).astype(np.float64)
    out_df["pattern_validates"] = agrees * vol_ok
    if CANDLE_CLASS_COL in out_df.columns:
        future = np.full(n, np.nan)
        future[: n - 1] = candle_ids[1:].astype(np.float64)
        if FUTURE_CANDLE_COL not in out_df.columns:
            out_df[FUTURE_CANDLE_COL] = future
    return out_df

def trim_v7_label_tail(
    df: pd.DataFrame,
    *,
    path_label_horizon_bars: int = PATH_LABEL_HORIZON_BARS,
    horizon_bars: Sequence[int] = HORIZON_BARS_5M,
) -> pd.DataFrame:
    """Drop rows without complete t+k and path labels."""
    max_h = max(int(path_label_horizon_bars), max(int(k) for k in horizon_bars))
    return df.iloc[: -(max_h + 1)].reset_index(drop=True)


## Inference and export helpers

In [ ]:
"""Window building and label un-standardization for per-TF transformer inference."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, Mapping, Sequence, Tuple

import numpy as np

def load_feature_config(path: Path) -> Dict[str, Any]:
    raw = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(raw, dict):
        raise ValueError(f"{path} must contain a JSON object")
    return raw

def resolve_feature_config(bundle_dir: Path) -> Dict[str, Any]:
    cfg_path = bundle_dir / TRANSFORMER_FEATURE_CONFIG_FILENAME
    if not cfg_path.is_file():
        raise FileNotFoundError(
            f"Missing {TRANSFORMER_FEATURE_CONFIG_FILENAME} in {bundle_dir}"
        )
    return load_feature_config(cfg_path)

def zscore_window(window: np.ndarray) -> np.ndarray:
    """Per-window z-score (matches Colab WindowDataset)."""
    mu = window.mean(axis=0, keepdims=True)
    sd = window.std(axis=0, keepdims=True) + 1e-6
    return ((window - mu) / sd).astype(np.float32)

def build_continuous_window(
    feat_values: np.ndarray,
    *,
    window_len: int,
    feature_cols: Sequence[str] = FEATURE_COLS,
) -> np.ndarray:
    """Build a z-scored (1, window_len, n_features) tensor from continuous cols."""
    if feat_values.shape[0] < window_len:
        raise ValueError(
            f"Need at least {window_len} feature rows, got {feat_values.shape[0]}"
        )
    window = feat_values[-window_len:, :].astype(np.float32)
    if not np.isfinite(window).all():
        raise ValueError("Feature window contains non-finite values")
    normed = zscore_window(window)
    return normed[np.newaxis, :, :]

def build_candle_class_window(
    class_ids: np.ndarray,
    *,
    window_len: int,
    max_class_id: int = CANDLE_CLASS_CARDINALITY - 1,
) -> np.ndarray:
    """Build a raw (1, window_len) int64 tensor of candle class ids."""
    ids = np.asarray(class_ids).reshape(-1)
    if ids.shape[0] < window_len:
        raise ValueError(
            f"Need at least {window_len} candle class rows, got {ids.shape[0]}"
        )
    window = ids[-window_len:].astype(np.int64)
    if np.any((window < 0) | (window > max_class_id)):
        raise ValueError(
            f"{CANDLE_CLASS_COL} out of range [0, {max_class_id}]"
        )
    return window[np.newaxis, :]

def build_inference_window(
    feat_values: np.ndarray,
    *,
    window_len: int,
    feature_cols: Sequence[str] = FEATURE_COLS,
) -> np.ndarray:
    """Build a single (1, window_len, n_features) tensor from feature matrix values."""
    return build_continuous_window(
        feat_values, window_len=window_len, feature_cols=feature_cols
    )

def unstandardize_continuous(
    pred_z: np.ndarray,
    label_mean: Sequence[float],
    label_std: Sequence[float],
    *,
    label_cols: Sequence[str] = CONTINUOUS_LABEL_COLS,
) -> Dict[str, float]:
    """Map standardized ONNX output back to real units."""
    mean = np.asarray(label_mean, dtype=np.float64)
    std = np.asarray(label_std, dtype=np.float64)
    flat = np.asarray(pred_z, dtype=np.float64).reshape(-1)
    if flat.shape[0] != len(label_cols):
        raise ValueError(
            f"Expected {len(label_cols)} continuous outputs, got {flat.shape[0]}"
        )
    real = flat * std + mean
    return {str(col): float(val) for col, val in zip(label_cols, real)}

def softmax(logits: np.ndarray) -> np.ndarray:
    x = np.asarray(logits, dtype=np.float64).reshape(-1)
    x = x - x.max()
    exp = np.exp(x)
    return exp / (exp.sum() + 1e-12)

def parse_regime_prediction(
    regime_logits: np.ndarray,
    regime_names: Mapping[str, str],
) -> Tuple[int, str, Dict[str, float]]:
    probs = softmax(regime_logits)
    idx = int(np.argmax(probs))
    name = regime_names.get(str(idx), regime_names.get(idx, f"CLASS_{idx}"))
    prob_map = {
        regime_names.get(str(i), f"CLASS_{i}"): float(probs[i])
        for i in range(len(probs))
    }
    return idx, str(name), prob_map

def require_onnx_output_names(
    output_names: Sequence[str],
    *,
    contract_version: str | None = None,
    resolution: str = "15m",
) -> None:
    """Reject bundles that lack the heads required by the given contract."""
    have = {str(name) for name in output_names}
    ver = str(contract_version or FEATURE_CONTRACT_VERSION_V6)
    required = onnx_output_names_for_contract(ver, resolution=resolution)
    known = (
        FEATURE_CONTRACT_VERSION,
        FEATURE_CONTRACT_VERSION_V8,
        FEATURE_CONTRACT_VERSION_V7,
        FEATURE_CONTRACT_VERSION_V6,
    )
    if ver not in known and not required:
        required = ONNX_OUTPUT_NAMES_V6
    missing = [name for name in required if name not in have]
    if missing:
        raise RuntimeError(
            f"ONNX bundle is not {ver}: missing outputs {missing}. Retrain all TFs."
        )

def feature_config_from_training_export(
    *,
    feature_cols: Sequence[str],
    window_len: int,
    label_mean: Sequence[float],
    label_std: Sequence[float],
    q_edges: Sequence[float],
    config: Mapping[str, Any],
    contract_version: str | None = None,
    onnx_output_names: Sequence[str] | None = None,
    continuous_label_cols: Sequence[str] | None = None,
) -> Dict[str, Any]:
    """Build feature_config.json payload written by Colab export cell."""
    version = str(contract_version or FEATURE_CONTRACT_VERSION_V6)
    if onnx_output_names is not None:
        names = list(onnx_output_names)
    elif version == FEATURE_CONTRACT_VERSION:
        names = list(ONNX_OUTPUT_NAMES_V9)
    elif version == FEATURE_CONTRACT_VERSION_V8:
        names = list(ONNX_OUTPUT_NAMES_V8)
    elif version == FEATURE_CONTRACT_VERSION_V7:
        names = list(ONNX_OUTPUT_NAMES_V7)
    else:
        names = list(ONNX_OUTPUT_NAMES)
    if continuous_label_cols is not None:
        label_cols = list(continuous_label_cols)
    elif version in (FEATURE_CONTRACT_VERSION, FEATURE_CONTRACT_VERSION_V8):
        label_cols = list(V8_CONTINUOUS_LABEL_COLS)
    else:
        label_cols = list(CONTINUOUS_LABEL_COLS)
    return {
        "feature_contract_version": version,
        "feature_cols": list(feature_cols),
        "categorical_cols": [CANDLE_CLASS_COL],
        "categorical_cardinality": {CANDLE_CLASS_COL: CANDLE_CLASS_CARDINALITY},
        "candle_class_names": {str(k): v for k, v in CANDLE_CLASS_NAMES.items()},
        "window_len": int(window_len),
        "continuous_label_cols": label_cols,
        "label_mean": [float(x) for x in label_mean],
        "label_std": [float(x) for x in label_std],
        "regime_names": {"0": "LOW", "1": "NORMAL", "2": "HIGH", "3": "EXTREME"},
        "structure_outcome_names": {
            str(k): v for k, v in STRUCTURE_OUTCOME_NAMES.items()
        },
        "onnx_output_names": names,
        "vol_regime_quantile_edges": [float(x) for x in q_edges],
        "config": dict(config),
    }

def metadata_from_training_export(
    *,
    resolution: str,
    label_mean: Sequence[float],
    label_std: Sequence[float],
    config: Mapping[str, Any],
    test_metrics: Mapping[str, Any] | None = None,
    export_quality: Mapping[str, Any] | None = None,
    onnx_output_names: Sequence[str] | None = None,
) -> Dict[str, Any]:
    """Build metadata_transformer.json for a per-TF bundle."""
    res = resolution.strip().lower()
    cfg = dict(config)
    names = list(onnx_output_names or ONNX_OUTPUT_NAMES)
    meta: Dict[str, Any] = {
        "version": "transformer_per_tf_v1",
        "model_name": f"jacksparrow_transformer_BTCUSD_{res}",
        "model_family": model_family_for_resolution(res),
        "symbol": str(cfg.get("symbol") or "BTCUSD"),
        "resolution": res,
        "resolution_minutes": int(cfg.get("resolution_minutes") or 15),
        "onnx_filename": onnx_filename_for_resolution(res),
        "feature_config_filename": TRANSFORMER_FEATURE_CONFIG_FILENAME,
        "path_label_horizon_bars": int(cfg.get("path_label_horizon_bars") or 8),
        "atr_period": int(cfg.get("atr_period") or 14),
        "default_threshold": float(cfg.get("default_threshold") or 0.005),
        "primary_signal_mode": "path_edge",
        "onnx_output_names": names,
        "label_mean": [float(x) for x in label_mean],
        "label_std": [float(x) for x in label_std],
        "test_metrics": dict(test_metrics or {}),
        "training_config": cfg,
    }
    if export_quality:
        meta["export_quality"] = dict(export_quality)
    return meta

def training_config_for_resolution(resolution: str) -> Dict[str, Any]:
    return default_training_config(resolution)


## Delta Exchange India data

In [ ]:
"""Public Delta India history fetch for Colab transformer training."""

from __future__ import annotations

import time
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional

import pandas as pd
import requests

_RESOLUTION_MINUTES = {
    "1m": 1,
    "3m": 3,
    "5m": 5,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
    "4h": 240,
    "1d": 1440,
}

# Documented candles cap is 2000; live India API has returned 4000 (newest-in-range
# when the window is larger). Request windows stay at the documented 2000. The loop
# walks backward from the oldest timestamp actually returned so a lower cap cannot
# open gaps. Production loaders use 500 for the same reason.
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95

def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(
            f"Unsupported resolution {resolution!r}; "
            f"use one of {sorted(_RESOLUTION_MINUTES)}"
        )
    return int(_RESOLUTION_MINUTES[resolution]) * 60

def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    """Normalize Delta history/candles payloads to a list of candle dicts."""
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    rows: List[Dict[str, Any]] = []
    for row in result:
        if isinstance(row, dict) and "time" in row:
            rows.append(row)
    return rows

def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None

def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []

def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df

def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    """Pull OHLC-style candles with response-driven backward pagination.

    Each request covers at most ``page_bars`` of wall-clock. The next ``end`` is the
    oldest timestamp actually returned minus one second, so a server cap below the
    requested window cannot skip bars.

    Args:
        symbol: Delta symbol (e.g. ``BTCUSD``, ``FUNDING:BTCUSD``).
        resolution: Candle resolution key in ``_RESOLUTION_MINUTES``.
        start_ts: Inclusive Unix seconds.
        end_ts: Exclusive-ish Unix seconds (API treats the range as a window).
        base_url: Exchange origin, without a trailing path.
        page_bars: Max bars per request window (default documented 2000).

    Returns:
        Deduplicated OHLCV frame sorted by ``time`` ascending.

    Raises:
        ValueError: Unknown resolution.
        RuntimeError: No candles in the requested range.
    """
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")

    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)

    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)

    if not all_rows:
        raise RuntimeError(
            f"No candle data returned for symbol={symbol}; "
            "check symbol/resolution/date range."
        )
    return _rows_to_frame(all_rows)

def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    """Measure gap rate on a native-TF OHLCV frame.

    Completeness is ``1 - gaps / (n - 1)`` where a gap is a bar-to-bar delta greater
    than ``1.5 * bar_seconds``. This catches sawtooth holes from truncated pages.

    Args:
        df: Frame with a ``time`` column (datetime or Unix seconds).
        resolution: Candle resolution key.
        min_completeness: Raise if the gap-free fraction is below this.
        symbol: Optional label for the error message.

    Returns:
        Report with ``rows``, ``gaps``, ``completeness``, and ``span_bars``.

    Raises:
        ValueError: Empty frame or completeness below ``min_completeness``.
    """
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    if pd.api.types.is_datetime64_any_dtype(time_col):
        epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    else:
        epochs = pd.to_numeric(time_col, errors="coerce").dropna().astype(int).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")

    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    span_bars = int((epochs[-1] - epochs[0]) // bar_seconds) + 1
    report: Dict[str, Any] = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
        "span_bars": span_bars,
        "expected_bars": span_bars,
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'} ({gaps} gaps, {len(epochs)} rows, "
            f"span {span_bars} bars). Refetch with response-driven pagination."
        )
    return report

def validate_derivatives_coverage(
    df: pd.DataFrame,
    *,
    min_coverage: float = 0.5,
    warn_coverage: float = 0.9,
) -> Dict[str, Any]:
    """Check funding/OI non-null coverage over the OHLCV timeline."""
    n = len(df)
    if n == 0:
        raise ValueError("Cannot validate derivatives coverage on empty frame")

    report: Dict[str, Any] = {"rows": n, "columns": {}}
    for col in ("funding_rate", "open_interest"):
        if col not in df.columns:
            report["columns"][col] = {"coverage": 0.0, "status": "missing"}
            continue
        coverage = float(df[col].notna().mean())
        status = "ok"
        if coverage < min_coverage:
            status = "error"
        elif coverage < warn_coverage:
            status = "warn"
        report["columns"][col] = {
            "coverage": round(coverage, 4),
            "status": status,
        }

    worst = min(v["coverage"] for v in report["columns"].values())
    report["worst_coverage"] = worst
    if worst < min_coverage:
        raise ValueError(
            "Derivatives coverage below minimum "
            f"({worst:.1%} < {min_coverage:.0%}): {report['columns']}"
        )
    return report

def fetch_history_bundle(
    *,
    symbol: str,
    resolution: str,
    history_days: int,
    base_url: str,
    min_derivatives_coverage: float = 0.5,
    derivatives_coverage_warn: float = 0.9,
    min_ohlcv_completeness: float = MIN_OHLCV_COMPLETENESS,
) -> pd.DataFrame:
    """Fetch OHLCV + funding + OI merged frame (notebook-compatible)."""
    end_dt = datetime.now(timezone.utc)
    start_dt = end_dt - timedelta(days=history_days)
    start_ts, end_ts = int(start_dt.timestamp()), int(end_dt.timestamp())

    raw_df = fetch_candles(symbol, resolution, start_ts, end_ts, base_url)
    ohlcv_report = validate_ohlcv_completeness(
        raw_df,
        resolution,
        min_completeness=min_ohlcv_completeness,
        symbol=symbol,
    )
    print(
        f"OHLCV {symbol} {resolution}: {ohlcv_report['rows']} bars, "
        f"completeness {ohlcv_report['completeness']:.1%}, "
        f"gaps={ohlcv_report['gaps']}"
    )

    try:
        funding_df = fetch_candles(
            f"FUNDING:{symbol}", resolution, start_ts, end_ts, base_url
        )
        funding_df = funding_df[["time", "close"]].rename(columns={"close": "funding_rate"})
    except Exception:
        funding_df = pd.DataFrame({"time": [], "funding_rate": []})

    try:
        oi_df = fetch_candles(f"OI:{symbol}", resolution, start_ts, end_ts, base_url)
        oi_df = oi_df[["time", "close"]].rename(columns={"close": "open_interest"})
    except Exception:
        oi_df = pd.DataFrame({"time": [], "open_interest": []})

    raw_df = assemble_raw_frame(raw_df, funding_df=funding_df, oi_df=oi_df)

    report = validate_derivatives_coverage(
        raw_df,
        min_coverage=min_derivatives_coverage,
        warn_coverage=derivatives_coverage_warn,
    )
    for col, info in report["columns"].items():
        if info["status"] == "warn":
            print(
                f"WARNING: {col} coverage {info['coverage']:.1%} "
                f"below recommended {derivatives_coverage_warn:.0%}"
            )
        elif info["status"] == "ok":
            print(f"{col} coverage: {info['coverage']:.1%}")

    return raw_df


## Multi-horizon model

In [ ]:
"""v9 multi-horizon 5m Transformer (research Colab; not the v6 path trainer)."""

from __future__ import annotations

import math
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset

def inverse_frequency_class_weights(
    class_ids: np.ndarray,
    n_classes: int,
) -> np.ndarray:
    """Inverse-frequency weights (mean-normalized) for imbalanced CE heads."""
    counts = np.bincount(
        np.asarray(class_ids, dtype=np.int64), minlength=int(n_classes)
    ).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = 1.0 / counts
    weights = weights * (float(n_classes) / weights.sum())
    return weights.astype(np.float32)

class NextCandleDataset(Dataset):
    """Windows of continuous features + candle ids with v9 horizon labels."""

    def __init__(
        self,
        x: np.ndarray,
        x_cat: np.ndarray,
        y_path: np.ndarray,
        path_mask: np.ndarray,
        volume_state: np.ndarray,
        horizon_dirs: np.ndarray,
        horizon_structs: np.ndarray,
        pattern_ids: np.ndarray | None = None,
        *,
        per_window_zscore: bool = False,
    ) -> None:
        self.x = x.astype(np.float32)
        self.x_cat = x_cat.astype(np.int64)
        self.y_path = y_path.astype(np.float32)
        self.path_mask = path_mask.astype(np.float32)
        self.volume_state = volume_state.astype(np.int64)
        self.horizon_dirs = horizon_dirs.astype(np.int64)
        self.horizon_structs = horizon_structs.astype(np.int64)
        if pattern_ids is None:
            pattern_ids = np.zeros(len(x), dtype=np.int64)
        self.pattern_ids = np.asarray(pattern_ids, dtype=np.int64)
        self.per_window_zscore = bool(per_window_zscore)

    def __len__(self) -> int:
        return int(len(self.x))

    def __getitem__(self, i: int) -> Tuple[torch.Tensor, ...]:
        window = self.x[i]
        if self.per_window_zscore:
            window = zscore_window(window)
        return (
            torch.tensor(window, dtype=torch.float32),
            torch.tensor(self.x_cat[i], dtype=torch.long),
            torch.tensor(self.y_path[i], dtype=torch.float32),
            torch.tensor(self.path_mask[i], dtype=torch.float32),
            torch.tensor(self.volume_state[i], dtype=torch.long),
            torch.tensor(self.horizon_dirs[i], dtype=torch.long),
            torch.tensor(self.horizon_structs[i], dtype=torch.long),
            torch.tensor(self.pattern_ids[i], dtype=torch.long),
        )

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512) -> None:
        super().__init__()
        self.pe = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]

class NextCandleTransformer(nn.Module):
    """Shared encoder with per-horizon direction, structure, path, and pattern heads."""

    def __init__(
        self,
        n_features: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dropout: float,
        max_len: int,
        n_continuous: int,
        n_horizons: int = N_HORIZONS,
        candle_embed_dim: int = CANDLE_EMBED_DIM,
    ) -> None:
        super().__init__()
        self.n_horizons = int(n_horizons)
        self.candle_embed = nn.Embedding(CANDLE_CLASS_CARDINALITY, candle_embed_dim)
        self.input_proj = nn.Linear(n_features + candle_embed_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        shared_dim = max(d_model // 2, 16)
        self.shared = nn.Sequential(
            nn.Linear(d_model, shared_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.dir_heads = nn.ModuleList(
            [
                nn.Linear(shared_dim, NEXT_DIRECTION_CARDINALITY)
                for _ in range(self.n_horizons)
            ]
        )
        self.structure_heads = nn.ModuleList(
            [
                nn.Linear(shared_dim, STRUCTURE_OUTCOME_CARDINALITY)
                for _ in range(self.n_horizons)
            ]
        )
        self.continuous_head = nn.Linear(shared_dim, n_continuous)
        self.volume_state_head = nn.Linear(shared_dim, VOLUME_STATE_CARDINALITY)
        self.pattern_head = nn.Linear(shared_dim, CHART_PATTERN_CARDINALITY)

    def forward(
        self, x: torch.Tensor, candle_class_ids: torch.Tensor
    ) -> Tuple[torch.Tensor, ...]:
        embed = self.candle_embed(candle_class_ids)
        h = self.input_proj(torch.cat([x, embed], dim=-1))
        h = self.pos_enc(h)
        h = self.encoder(h)
        h = self.norm(h)
        shared = self.shared(h[:, -1, :])
        dir_outs = tuple(head(shared) for head in self.dir_heads)
        struct_outs = tuple(head(shared) for head in self.structure_heads)
        return (
            *dir_outs,
            *struct_outs,
            self.continuous_head(shared),
            self.volume_state_head(shared),
            self.pattern_head(shared),
        )

def compute_v9_loss(
    outputs: Sequence[torch.Tensor],
    batch: Sequence[torch.Tensor],
    *,
    loss_weights: Dict[str, float] | None = None,
    dir_class_weights: Optional[torch.Tensor] = None,
    pattern_class_weights: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Per-horizon path MSE + direction/structure CE + volume/pattern CE."""
    weights = dict(V9_STRUCTURE_LOSS_WEIGHTS)
    if loss_weights:
        weights.update({str(k): float(v) for k, v in loss_weights.items()})
    volume_y = batch[4]
    horizon_dirs = batch[5]
    horizon_structs = batch[6]
    pattern_y = batch[7] if len(batch) > 7 else None
    y_path = batch[2]
    path_mask = batch[3]
    n_h = int(horizon_dirs.size(1))
    dir_logits = outputs[:n_h]
    struct_logits = outputs[n_h : 2 * n_h]
    cont_pred = outputs[2 * n_h]
    vol_logits = outputs[2 * n_h + 1]
    pattern_logits = outputs[2 * n_h + 2] if len(outputs) > 2 * n_h + 2 else None

    def _ce_mean(
        logits: torch.Tensor,
        target: torch.Tensor,
        w: float,
        class_w: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        return w * nn.functional.cross_entropy(
            logits, target, weight=class_w, reduction="mean"
        )

    n_cont = min(
        cont_pred.size(1), y_path.size(1), len(V8_CONTINUOUS_LABEL_COLS)
    )
    path_diff = (cont_pred[:, :n_cont] - y_path[:, :n_cont]) ** 2
    masked = (path_diff * path_mask[:, :n_cont]).sum() / (
        path_mask[:, :n_cont].sum() + 1e-6
    )
    loss = float(weights["path"]) * masked
    loss = loss + _ce_mean(vol_logits, volume_y, float(weights["volume_state"]))

    dir_w = float(weights["direction"]) / max(n_h, 1)
    struct_w = float(weights["structure"]) / max(n_h, 1)
    for j in range(n_h):
        loss = loss + _ce_mean(
            dir_logits[j], horizon_dirs[:, j], dir_w, dir_class_weights
        )
        loss = loss + _ce_mean(struct_logits[j], horizon_structs[:, j], struct_w)
    if pattern_logits is not None and pattern_y is not None:
        loss = loss + _ce_mean(
            pattern_logits,
            pattern_y,
            float(weights.get("pattern", 0.25)),
            pattern_class_weights,
        )
    return loss

def compute_v8_loss(
    outputs: Sequence[torch.Tensor],
    batch: Sequence[torch.Tensor],
    *,
    loss_weights: Dict[str, float] | None = None,
    dir_class_weights: Optional[torch.Tensor] = None,
    pattern_class_weights: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Alias kept so older notebook cells/tests can import a loss name."""
    return compute_v9_loss(
        outputs,
        batch,
        loss_weights=loss_weights,
        dir_class_weights=dir_class_weights,
        pattern_class_weights=pattern_class_weights,
    )

def compute_v7_loss(
    outputs: Sequence[torch.Tensor],
    batch: Sequence[torch.Tensor],
    *,
    loss_weights: Dict[str, float] | None = None,
) -> torch.Tensor:
    """Alias kept so older notebook cells/tests can import a loss name."""
    return compute_v9_loss(outputs, batch, loss_weights=loss_weights)

def train_next_candle(
    model: NextCandleTransformer,
    train_loader: torch.utils.data.DataLoader,
    val_loader: torch.utils.data.DataLoader,
    *,
    device: torch.device,
    epochs: int,
    lr: float,
    weight_decay: float,
    patience: int,
    loss_weights: Dict[str, float] | None = None,
    dir_class_weights: Optional[torch.Tensor] = None,
    pattern_class_weights: Optional[torch.Tensor] = None,
) -> Dict[str, Any]:
    """Train with AdamW and early stopping on validation loss."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val = float("inf")
    best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    stale = 0
    history: List[Tuple[int, float, float]] = []
    aborted = False
    dir_w = (
        dir_class_weights.to(device)
        if dir_class_weights is not None
        else None
    )
    pat_w = (
        pattern_class_weights.to(device)
        if pattern_class_weights is not None
        else None
    )
    for epoch in range(int(epochs)):
        model.train()
        train_loss = 0.0
        n_train = 0
        for batch in train_loader:
            batch_d = tuple(t.to(device) for t in batch)
            opt.zero_grad(set_to_none=True)
            outs = model(batch_d[0], batch_d[1])
            loss = compute_v9_loss(
                outs,
                batch_d,
                loss_weights=loss_weights,
                dir_class_weights=dir_w,
                pattern_class_weights=pat_w,
            )
            loss_val = float(loss.detach().item())
            if not math.isfinite(loss_val):
                print(f"Non-finite train loss at epoch {epoch + 1} — aborting")
                aborted = True
                break
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_loss += loss_val
            n_train += 1
        if aborted:
            break
        if n_train == 0:
            print(f"No finite train batches at epoch {epoch + 1} — aborting")
            aborted = True
            break
        model.eval()
        val_loss = 0.0
        n_val = 0
        with torch.no_grad():
            for batch in val_loader:
                batch_d = tuple(t.to(device) for t in batch)
                outs = model(batch_d[0], batch_d[1])
                batch_val = float(
                    compute_v9_loss(
                        outs,
                        batch_d,
                        loss_weights=loss_weights,
                        dir_class_weights=dir_w,
                        pattern_class_weights=pat_w,
                    ).item()
                )
                if not math.isfinite(batch_val):
                    print(f"Non-finite val loss at epoch {epoch + 1} — aborting")
                    aborted = True
                    break
                val_loss += batch_val
                n_val += 1
        if aborted:
            break
        train_loss /= max(n_train, 1)
        val_loss /= max(n_val, 1)
        history.append((epoch + 1, train_loss, val_loss))
        print(f"epoch {epoch + 1:03d}  train={train_loss:.4f}  val={val_loss:.4f}")
        if not math.isfinite(train_loss) or not math.isfinite(val_loss):
            print(f"Non-finite epoch loss at {epoch + 1} — aborting")
            aborted = True
            break
        if val_loss < best_val:
            best_val = val_loss
            stale = 0
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= int(patience):
                print(f"Early stopping at epoch {epoch + 1}")
                break
    model.load_state_dict(best_state)
    epochs_ran = float(history[-1][0]) if history else 0.0
    ok = (not aborted) and math.isfinite(best_val) and bool(history)
    return {
        "ok": ok,
        "best_val_loss": float(best_val) if math.isfinite(best_val) else float("inf"),
        "epochs_ran": epochs_ran,
    }


## Multi-horizon research pipeline

In [ ]:
"""Research pipeline: multi-horizon 5m path training for the Transformer."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

FUTURE_LEAK_COLS = v8_future_leak_cols()

def set_research_seed(seed: int) -> None:
    """Seed numpy and torch for Colab reproducibility."""
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

def ohlcv_quality_report(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = 0.95,
    symbol: str = "BTCUSD",
) -> Dict[str, Any]:
    """Timestamp, duplicate, OHLC-validity, and gap report. Does not mutate OHLC."""
    if df is None or df.empty:
        raise ValueError("OHLCV frame is empty")
    frame = df.copy()
    if "time" not in frame.columns:
        raise ValueError("OHLCV requires a time column")
    frame["time"] = pd.to_datetime(frame["time"], utc=True)
    n = len(frame)
    dupes = int(frame["time"].duplicated().sum())
    unsorted = not bool(frame["time"].is_monotonic_increasing)
    invalid = 0
    if all(c in frame.columns for c in ("open", "high", "low", "close")):
        invalid = int(
            (
                (frame["high"] < frame["low"])
                | (frame["high"] < frame["open"])
                | (frame["high"] < frame["close"])
                | (frame["low"] > frame["open"])
                | (frame["low"] > frame["close"])
            ).sum()
        )
    neg_vol = 0
    if "volume" in frame.columns:
        neg_vol = int((frame["volume"] < 0).sum())
    nan_cells = int(frame.isna().sum().sum())
    sorted_t = frame.sort_values("time")["time"]
    deltas = sorted_t.diff().dt.total_seconds().dropna()
    bar_sec = float(_bar_seconds(resolution))
    missing = int(np.maximum((deltas / bar_sec) - 1.0, 0.0).round().sum())
    completeness = validate_ohlcv_completeness(
        frame.sort_values("time"),
        resolution,
        min_completeness=min_completeness,
        symbol=symbol,
    )
    report = {
        "rows": n,
        "time_start": str(frame["time"].min()),
        "time_end": str(frame["time"].max()),
        "duplicates": dupes,
        "unsorted": unsorted,
        "invalid_ohlc": invalid,
        "negative_volume": neg_vol,
        "nans": nan_cells,
        "missing_candles": missing if missing else completeness.get("gaps", 0),
        "completeness": completeness.get("completeness"),
    }
    print(
        "OHLCV quality: "
        f"rows={n} range={report['time_start']} → {report['time_end']} "
        f"missing={report['missing_candles']} dupes={dupes} "
        f"invalid_ohlc={invalid} nans={nan_cells}"
    )
    if dupes or invalid or neg_vol:
        raise ValueError(f"OHLCV quality failed: {report}")
    return report

def leakage_audit(feature_cols: Sequence[str]) -> None:
    """Raise if any target/future column leaked into the input feature list."""
    leaked = [c for c in feature_cols if c in FUTURE_LEAK_COLS]
    if leaked:
        raise RuntimeError(f"Future/target columns in inputs: {leaked}")

def chronological_split(
    n: int,
    *,
    train_ratio: float,
    validation_ratio: float,
    embargo: int,
) -> Dict[str, slice]:
    """Time-ordered train/val/test slices with an embargo gap.

    Embargo shrinks when the split would otherwise empty val or test.
    """
    n = int(n)
    train_end = max(1, int(n * float(train_ratio)))
    val_span = max(1, int(n * float(validation_ratio)))
    val_end = min(n, train_end + val_span)
    if val_end <= train_end:
        val_end = min(n, train_end + 1)
    gap = max(int(embargo), 0)
    gap = min(gap, max(0, val_end - train_end - 1), max(0, n - val_end - 1))
    val_start = train_end + gap
    test_start = val_end + gap
    if val_start >= val_end:
        val_start = train_end
    if test_start >= n:
        test_start = val_end
    return {
        "train": slice(0, train_end),
        "val": slice(val_start, val_end),
        "test": slice(test_start, n),
    }

def sanitize_feature_values(values: np.ndarray) -> np.ndarray:
    """Replace inf with NaN, then fill remaining NaNs with 0. Always finite."""
    cleaned = np.asarray(values, dtype=np.float64).copy()
    cleaned[~np.isfinite(cleaned)] = np.nan
    return np.nan_to_num(cleaned, nan=0.0, posinf=0.0, neginf=0.0)

def feature_finite_report(
    values: np.ndarray,
    feature_cols: Optional[Sequence[str]] = None,
) -> Dict[str, Any]:
    """Print finite-rate and per-column inf/nan counts before sanitizing."""
    arr = np.asarray(values, dtype=np.float64)
    finite_rate = float(np.isfinite(arr).mean()) if arr.size else 1.0
    n_inf = int(np.isinf(arr).sum())
    n_nan = int(np.isnan(arr).sum())
    bad_cols: List[Dict[str, Any]] = []
    if arr.ndim == 2 and arr.shape[1] > 0:
        names: Sequence[str]
        if feature_cols is not None and len(feature_cols) == arr.shape[1]:
            names = feature_cols
        else:
            names = [f"c{i}" for i in range(arr.shape[1])]
        for i, name in enumerate(names):
            col = arr[:, i]
            c_inf = int(np.isinf(col).sum())
            c_nan = int(np.isnan(col).sum())
            if c_inf or c_nan:
                bad_cols.append({"col": str(name), "inf": c_inf, "nan": c_nan})
    report = {
        "finite_rate": finite_rate,
        "inf": n_inf,
        "nan": n_nan,
        "bad_cols": bad_cols,
    }
    print(
        f"feature finite_rate={finite_rate:.6f} inf={n_inf} nan={n_nan} "
        f"bad_cols={len(bad_cols)}"
    )
    for item in bad_cols[:12]:
        print(f"  {item['col']}: inf={item['inf']} nan={item['nan']}")
    if len(bad_cols) > 12:
        print(f"  ... {len(bad_cols) - 12} more columns")
    return report

def fit_label_stats(y_train: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Train-split mean/std for path-label standardization (NaNs excluded)."""
    mean = np.nan_to_num(np.nanmean(y_train, axis=0), nan=0.0)
    std = np.nan_to_num(np.nanstd(y_train, axis=0), nan=1.0) + 1e-9
    return mean.astype(np.float64), std.astype(np.float64)

def standardize_labels(
    y: np.ndarray,
    label_mean: np.ndarray,
    label_std: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Z-score path labels and build per-target validity masks."""
    mask = (~np.isnan(y)).astype(np.float32)
    yz = np.where(np.isnan(y), 0.0, (y - label_mean) / label_std).astype(np.float32)
    return yz, mask

def fit_train_scaler(
    train_values: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Column mean/std from TRAIN rows only."""
    cleaned = sanitize_feature_values(train_values)
    mean = np.nanmean(cleaned, axis=0)
    std = np.nanstd(cleaned, axis=0) + 1e-6
    return mean.astype(np.float32), std.astype(np.float32)

def apply_scaler(
    values: np.ndarray,
    mean: np.ndarray,
    std: np.ndarray,
) -> np.ndarray:
    """Apply a frozen train scaler. Does not refit."""
    return ((values - mean) / std).astype(np.float32)

def build_labeled_frame(
    raw: pd.DataFrame,
    *,
    config: Mapping[str, Any],
) -> pd.DataFrame:
    """Causal features + v8 multi-horizon behavior labels. Raw OHLCV is copied."""
    resolution_minutes = int(config.get("resolution_minutes") or 5)
    atr_period = int(config.get("atr_period") or 14)
    feat = add_features(
        assemble_raw_frame(raw),
        resolution_minutes=resolution_minutes,
        atr_period=atr_period,
    )
    labeled = compute_horizon_behavior_labels(feat)
    return trim_v8_label_tail(labeled)

def _stack_windows(
    values: np.ndarray,
    ids: np.ndarray,
    window_len: int,
    stride: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    x_list: List[np.ndarray] = []
    cat_list: List[np.ndarray] = []
    ends: List[int] = []
    for end in range(window_len, len(values), stride):
        start = end - window_len
        x_list.append(values[start:end])
        cat_list.append(ids[start:end])
        ends.append(end - 1)
    return (
        np.array(x_list, dtype=np.float32),
        np.array(cat_list, dtype=np.int64),
        np.array(ends, dtype=np.int64),
    )

def windows_from_frame(
    df: pd.DataFrame,
    *,
    feature_cols: Sequence[str],
    window_len: int,
    stride: int,
    scaler_mean: Optional[np.ndarray] = None,
    scaler_std: Optional[np.ndarray] = None,
    per_window_zscore: bool = False,
) -> Dict[str, np.ndarray]:
    """Build overlapping windows with labels taken at the last bar of each window."""
    leakage_audit(feature_cols)
    values = sanitize_feature_values(df[list(feature_cols)].to_numpy(dtype=np.float64))
    if scaler_mean is not None and scaler_std is not None:
        values = sanitize_feature_values(apply_scaler(values, scaler_mean, scaler_std))
    ids = (
        df[CANDLE_CLASS_COL].to_numpy(dtype=np.int64)
        if CANDLE_CLASS_COL in df.columns
        else np.zeros(len(df), dtype=np.int64)
    )
    x, x_cat, idx = _stack_windows(values, ids, window_len, stride)

    def _take(col: str, default: float = 0.0) -> np.ndarray:
        if col not in df.columns:
            return np.full(len(idx), default)
        arr = df[col].to_numpy()[idx]
        return np.nan_to_num(arr, nan=default)

    path = np.column_stack(
        [_take(c, np.nan) for c in V8_CONTINUOUS_LABEL_COLS]
    ).astype(np.float64)
    horizon_dirs = np.column_stack(
        [_take(c, 2.0) for c in HORIZON_DIR_COLS]
    ).astype(np.int64)
    horizon_structs = np.column_stack(
        [_take(c, 0.0) for c in HORIZON_STRUCTURE_COLS]
    ).astype(np.int64)
    pattern_ids = np.clip(
        _take(CHART_PATTERN_COL, 0.0).astype(np.int64),
        0,
        CHART_PATTERN_CARDINALITY - 1,
    )
    return {
        "x": x,
        "x_cat": x_cat,
        "idx": idx,
        "y_path": path,
        "volume_state": _take(VOLUME_STATE_COL, 1.0).astype(np.int64),
        "horizon_dirs": np.clip(horizon_dirs, 0, NEXT_DIRECTION_CARDINALITY - 1),
        "horizon_structs": np.clip(horizon_structs, 0, 5),
        "pattern_ids": pattern_ids,
        "per_window_zscore": np.array([per_window_zscore], dtype=np.bool_),
    }

def split_window_dict(
    packed: Dict[str, np.ndarray],
    slices: Mapping[str, slice],
) -> Dict[str, Dict[str, np.ndarray]]:
    """Apply chronological slices to packed window arrays."""
    skip = {"per_window_zscore"}
    n = len(packed["x"])
    out: Dict[str, Dict[str, np.ndarray]] = {}
    for name, sl in slices.items():
        start = max(0, sl.start or 0)
        stop = min(n, sl.stop if sl.stop is not None else n)
        if stop <= start:
            raise ValueError(f"Empty {name} split after embargo")
        out[name] = {
            k: (v if k in skip else v[start:stop])
            for k, v in packed.items()
        }
    return out

def make_loader(
    packed: Dict[str, np.ndarray],
    *,
    y_path_z: np.ndarray,
    path_mask: np.ndarray,
    batch_size: int,
    shuffle: bool,
    per_window_zscore: bool,
) -> DataLoader:
    ds = NextCandleDataset(
        packed["x"],
        packed["x_cat"],
        y_path_z,
        path_mask,
        packed["volume_state"],
        packed["horizon_dirs"],
        packed["horizon_structs"],
        packed.get("pattern_ids", np.zeros(len(packed["x"]), dtype=np.int64)),
        per_window_zscore=per_window_zscore,
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle)

def classification_report_head(
    pred: np.ndarray,
    true: np.ndarray,
    *,
    name: str,
) -> Dict[str, float]:
    """Accuracy / balanced accuracy without requiring sklearn at import time."""
    pred = np.asarray(pred, dtype=np.int64)
    true = np.asarray(true, dtype=np.int64)
    mask = true >= 0
    pred = pred[mask]
    true = true[mask]
    if len(true) == 0:
        return {"name": name, "accuracy": 0.0, "balanced_accuracy": 0.0, "n": 0}
    acc = float((pred == true).mean())
    classes = np.unique(true)
    recalls = []
    for c in classes:
        denom = float((true == c).sum())
        if denom <= 0:
            continue
        recalls.append(float(((pred == c) & (true == c)).sum()) / denom)
    bal = float(np.mean(recalls)) if recalls else acc
    print(f"  {name:16s} acc={acc:.3f}  balanced={bal:.3f}  n={len(true)}")
    return {"name": name, "accuracy": acc, "balanced_accuracy": bal, "n": float(len(true))}

def _tensor_to_numpy(tensor: torch.Tensor) -> np.ndarray:
    """Convert without the torch–NumPy C-API (missing in some CPU wheels)."""
    return np.asarray(tensor.detach().cpu().tolist())

@torch.no_grad()
def evaluate_structure_heads(
    model: NextCandleTransformer,
    loader: DataLoader,
    *,
    device: torch.device,
) -> Dict[str, Any]:
    """Collect per-horizon direction accuracy and path-vol correlation."""
    model.eval()
    dir_pred: List[np.ndarray] = []
    dir_true: List[np.ndarray] = []
    path_pred: List[np.ndarray] = []
    path_true: List[np.ndarray] = []
    n_h = N_HORIZONS
    for batch in loader:
        xb, xcat = batch[0].to(device), batch[1].to(device)
        outs = model(xb, xcat)
        dirs = np.stack(
            [_tensor_to_numpy(outs[j].argmax(dim=1)) for j in range(n_h)],
            axis=1,
        )
        dir_pred.append(dirs)
        dir_true.append(_tensor_to_numpy(batch[5]))
        path_pred.append(_tensor_to_numpy(outs[2 * n_h]))
        path_true.append(_tensor_to_numpy(batch[2]))
    pred_d = np.concatenate(dir_pred, axis=0)
    true_d = np.concatenate(dir_true, axis=0)
    reports: Dict[str, Any] = {}
    for j, key in enumerate(HORIZON_KEYS):
        reports[f"{key}_dir"] = classification_report_head(
            pred_d[:, j], true_d[:, j], name=f"{key}_dir"
        )
    reports["direction"] = reports["h5m_dir"]
    pred_p = np.concatenate(path_pred, axis=0)
    true_p = np.concatenate(path_true, axis=0)
    vol_idx = [
        i for i, col in enumerate(V8_CONTINUOUS_LABEL_COLS) if col.endswith("_vol")
    ]
    corrs: List[float] = []
    for idx in vol_idx:
        a = pred_p[:, idx]
        b = true_p[:, idx]
        mask = np.isfinite(a) & np.isfinite(b)
        if int(mask.sum()) < 8:
            continue
        if float(np.std(a[mask])) < 1e-12 or float(np.std(b[mask])) < 1e-12:
            continue
        corrs.append(float(np.corrcoef(a[mask], b[mask])[0, 1]))
    reports["path_vol_corr"] = float(np.mean(corrs)) if corrs else 0.0
    print(f"  path_vol_corr={reports['path_vol_corr']:.3f}")
    return reports

def walk_forward_slices(
    n: int,
    *,
    folds: int,
    embargo: int,
) -> List[Tuple[slice, slice]]:
    """Expanding train, trailing val; never includes a held-out final test."""
    folds = max(int(folds), 1)
    out: List[Tuple[slice, slice]] = []
    for i in range(folds):
        train_end = max(2, int(n * (0.50 + 0.10 * i)))
        val_end = min(n, train_end + max(int(n * 0.12), 8))
        gap = min(max(int(embargo), 0), max(0, val_end - train_end - 1))
        val_start = train_end + gap
        if val_start >= val_end:
            continue
        out.append((slice(0, train_end), slice(val_start, val_end)))
    return out

def confusion_counts(pred: np.ndarray, true: np.ndarray, n_classes: int) -> np.ndarray:
    """Integer confusion matrix (rows=true, cols=pred)."""
    mat = np.zeros((n_classes, n_classes), dtype=np.int64)
    pred = np.asarray(pred, dtype=np.int64)
    true = np.asarray(true, dtype=np.int64)
    for t, p in zip(true, pred):
        if 0 <= t < n_classes and 0 <= p < n_classes:
            mat[t, p] += 1
    return mat

def direction_class_mix_report(df: pd.DataFrame) -> Dict[str, Any]:
    """Print 5-class mix and |close[t+k]-close[t]| / ATR quantiles per horizon."""

    report: Dict[str, Any] = {}
    print("Direction class mix (0=STRONG_DOWN ... 4=STRONG_UP):")
    for col in HORIZON_DIR_COLS:
        if col not in df.columns:
            continue
        counts = df[col].value_counts(dropna=True).sort_index().to_dict()
        named = {
            NEXT_DIRECTION_NAMES.get(int(k), str(k)): int(v) for k, v in counts.items()
        }
        report[col] = named
        print(col, named)
    if "close" in df.columns and "atr" in df.columns:
        close = df["close"].to_numpy(dtype=np.float64)
        atr = np.maximum(df["atr"].to_numpy(dtype=np.float64), 1e-9)
        print("|move|/ATR quantiles:")
        for key, k in HORIZON_SPECS:
            ratios: List[float] = []
            n = len(close)
            kk = int(k)
            for i in range(n - kk):
                if not np.isfinite(close[i]) or not np.isfinite(close[i + kk]):
                    continue
                ratios.append(abs(float(close[i + kk] - close[i])) / float(atr[i]))
            if not ratios:
                continue
            arr = np.asarray(ratios, dtype=np.float64)
            qs = {
                "p50": float(np.quantile(arr, 0.50)),
                "p90": float(np.quantile(arr, 0.90)),
                "p99": float(np.quantile(arr, 0.99)),
            }
            report[f"{key}_abs_move_atr"] = qs
            print(f"  {key}: p50={qs['p50']:.3f} p90={qs['p90']:.3f} p99={qs['p99']:.3f}")
    return report

def pattern_context_table(df: pd.DataFrame) -> pd.DataFrame:
    """Named pattern × trend/volume vs subsequent 5m path_edge (analysis only)."""
    mfe_col = "h5m_mfe" if "h5m_mfe" in df.columns else "mfe"
    mae_col = "h5m_mae" if "h5m_mae" in df.columns else "mae"
    need = [
        c
        for c in (CHART_PATTERN_COL, "structure_bias", "vol_z", mfe_col, mae_col)
        if c in df.columns
    ]
    if len(need) < 3:
        return pd.DataFrame()
    work = df[need].copy()
    if mfe_col in work.columns and mae_col in work.columns:
        work["path_edge"] = work[mfe_col] - work[mae_col]
    grouped = work.groupby(CHART_PATTERN_COL, dropna=True).agg(
        n=(need[1], "count"),
        path_edge_mean=(
            "path_edge",
            "mean",
        )
        if "path_edge" in work.columns
        else (need[1], "mean"),
        structure_bias_mean=(
            "structure_bias",
            "mean",
        )
        if "structure_bias" in work.columns
        else (need[1], "mean"),
    )
    print(grouped.to_string())
    return grouped

def shap_grouped_stub(feature_cols: Sequence[str], *, enabled: bool) -> Dict[str, List[str]]:
    """Print grouped feature names for SHAP; skip computation unless enabled."""
    groups = {
        "candle": [c for c in feature_cols if c in (
            "body_ratio", "upper_wick_ratio", "lower_wick_ratio", "close_loc",
            "range_atr", "body_atr", "gap_atr", "inside_bar", "outside_bar",
        )],
        "trend": [c for c in feature_cols if "ema" in c or c in ("macd_hist", "adx_14", "rsi_14")],
        "structure": [c for c in feature_cols if c.startswith(("hh_", "hl_", "lh_", "ll_", "structure_"))],
        "chart_geometry": [c for c in feature_cols if c in (
            "peak_diff_atr", "trough_diff_atr", "flag_width_atr", CHART_PATTERN_COL,
        )],
        "vol": [c for c in feature_cols if c.startswith("rv_") or c == "atr"],
        "volume": [c for c in feature_cols if "vol" in c],
        "sr": [c for c in feature_cols if "support" in c or "resistance" in c],
        "multi_tf": [c for c in feature_cols if c.startswith("htf_")],
    }
    if not enabled:
        print("SHAP skipped (CONFIG run_shap=False). Feature groups:")
        for name, cols in groups.items():
            print(f"  {name}: {len(cols)} cols")
        return groups
    print("SHAP enabled — install shap in the Colab env and call shap.Explainer on a loader.")
    return groups

def optuna_search_stub(config: Mapping[str, Any]) -> Dict[str, Any]:
    """Optuna on walk-forward validation only. Never uses the final test split."""
    cfg = dict(config)
    if not cfg.get("run_optuna"):
        print("Optuna skipped (CONFIG run_optuna=False). Search space: sequence_length, lr, "
              "weight_decay, dropout, layers, heads, d_model, batch, loss lambdas.")
        return cfg
    try:
        import optuna  # noqa: F401
    except ImportError:
        print("Optuna not installed; leaving CONFIG unchanged.")
        return cfg
    print("Optuna enabled — objective must be walk-forward validation loss, never test.")
    return cfg

def run_walk_forward_eval(
    packed: Dict[str, np.ndarray],
    *,
    config: Mapping[str, Any],
    device: torch.device,
    y_mean: np.ndarray,
    y_std: np.ndarray,
    feature_index: Optional[np.ndarray] = None,
) -> List[float]:
    """One-epoch expanding-window direction accuracy (validation folds only)."""
    n = len(packed["x"])
    embargo = max(1, int(config.get("walk_forward_embargo") or 1))
    scores: List[float] = []
    for train_sl, val_sl in walk_forward_slices(
        n, folds=int(config.get("walk_forward_folds") or 3), embargo=embargo
    ):
        tr = {k: (v[train_sl] if k != "per_window_zscore" else v) for k, v in packed.items()}
        va = {k: (v[val_sl] if k != "per_window_zscore" else v) for k, v in packed.items()}
        if feature_index is not None:
            tr["x"] = tr["x"][:, :, feature_index]
            va["x"] = va["x"][:, :, feature_index]
        if len(tr["x"]) < 4 or len(va["x"]) < 2:
            continue
        scores.append(
            run_ablation_epoch(
                tr, va, feature_index=np.arange(tr["x"].shape[-1]),
                config=config, device=device, y_mean=y_mean, y_std=y_std,
            )
        )
    print(f"Walk-forward val direction acc: {scores}")
    return scores

def run_ablation_epoch(
    packed_train: Dict[str, np.ndarray],
    packed_val: Dict[str, np.ndarray],
    *,
    feature_index: np.ndarray,
    config: Mapping[str, Any],
    device: torch.device,
    y_mean: np.ndarray,
    y_std: np.ndarray,
) -> float:
    """Single-epoch ablation on a feature subset; returns val direction accuracy."""
    def _sub(p: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
        q = dict(p)
        q["x"] = p["x"][:, :, feature_index]
        return q

    tr = _sub(packed_train)
    va = _sub(packed_val)
    ytr_z, mtr = standardize_labels(tr["y_path"], y_mean, y_std)
    yva_z, mva = standardize_labels(va["y_path"], y_mean, y_std)
    model = NextCandleTransformer(
        n_features=int(tr["x"].shape[-1]),
        d_model=32,
        nhead=4,
        num_layers=1,
        dropout=0.1,
        max_len=int(config.get("sequence_length") or 64),
        n_continuous=len(V8_CONTINUOUS_LABEL_COLS),
    ).to(device)
    loader_tr = make_loader(
        tr,
        y_path_z=ytr_z,
        path_mask=mtr,
        batch_size=min(64, len(tr["x"])),
        shuffle=True,
        per_window_zscore=False,
    )
    loader_va = make_loader(
        va,
        y_path_z=yva_z,
        path_mask=mva,
        batch_size=min(64, max(len(va["x"]), 1)),
        shuffle=False,
        per_window_zscore=False,
    )
    train_next_candle(
        model,
        loader_tr,
        loader_va,
        device=device,
        epochs=1,
        lr=1e-3,
        weight_decay=1e-4,
        patience=1,
        loss_weights=config.get("loss_weights"),
    )
    metrics = evaluate_structure_heads(model, loader_va, device=device)
    return float(metrics["direction"]["accuracy"])

def export_v9_bundle(
    model: NextCandleTransformer,
    export_dir: Path,
    *,
    device: torch.device,
    window_len: int,
    n_features: int,
    feature_cols: Sequence[str],
    label_mean: np.ndarray,
    label_std: np.ndarray,
    config: Mapping[str, Any],
    scaler_mean: Optional[np.ndarray] = None,
    scaler_std: Optional[np.ndarray] = None,
) -> Tuple[Path, Path, Path]:
    """Write ONNX + feature_config.json + metadata_transformer.json (v9)."""
    export_dir.mkdir(parents=True, exist_ok=True)
    onnx_path = export_dir / "btcusd_5m_transformer.onnx"
    dummy_x = torch.zeros(1, window_len, n_features, device=device)
    dummy_c = torch.zeros(1, window_len, dtype=torch.long, device=device)
    model.eval()
    output_names = list(ONNX_OUTPUT_NAMES_V9)
    torch.onnx.export(
        model,
        (dummy_x, dummy_c),
        str(onnx_path),
        input_names=["continuous_features", "candle_class_ids"],
        output_names=output_names,
        dynamic_axes={
            "continuous_features": {0: "batch"},
            "candle_class_ids": {0: "batch"},
            **{name: {0: "batch"} for name in output_names},
        },
        opset_version=17,
        dynamo=False,
    )
    cfg = dict(config)
    feature_config = feature_config_from_training_export(
        feature_cols=feature_cols,
        window_len=window_len,
        label_mean=label_mean,
        label_std=label_std,
        q_edges=[0.25, 0.5, 0.75],
        config=cfg,
        contract_version=FEATURE_CONTRACT_VERSION,
        onnx_output_names=output_names,
        continuous_label_cols=V8_CONTINUOUS_LABEL_COLS,
    )
    if scaler_mean is not None and scaler_std is not None:
        feature_config["scaler_mean"] = [float(x) for x in scaler_mean]
        feature_config["scaler_std"] = [float(x) for x in scaler_std]
        feature_config["scaler_mode"] = "train_fit"
    cfg_path = export_dir / "feature_config.json"
    cfg_path.write_text(json.dumps(feature_config, indent=2), encoding="utf-8")
    meta = metadata_from_training_export(
        resolution="5m",
        label_mean=label_mean,
        label_std=label_std,
        config=cfg,
        onnx_output_names=output_names,
    )
    meta_path = export_dir / "metadata_transformer.json"
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return onnx_path, cfg_path, meta_path

def export_v8_bundle(
    model: NextCandleTransformer,
    export_dir: Path,
    *,
    device: torch.device,
    window_len: int,
    n_features: int,
    feature_cols: Sequence[str],
    label_mean: np.ndarray,
    label_std: np.ndarray,
    config: Mapping[str, Any],
    scaler_mean: Optional[np.ndarray] = None,
    scaler_std: Optional[np.ndarray] = None,
) -> Tuple[Path, Path, Path]:
    """Alias: research export is v9 (kept so older notebook cells still import)."""
    return export_v9_bundle(
        model,
        export_dir,
        device=device,
        window_len=window_len,
        n_features=n_features,
        feature_cols=feature_cols,
        label_mean=label_mean,
        label_std=label_std,
        config=config,
        scaler_mean=scaler_mean,
        scaler_std=scaler_std,
    )

def export_v7_bundle(
    model: NextCandleTransformer,
    export_dir: Path,
    *,
    device: torch.device,
    window_len: int,
    n_features: int,
    feature_cols: Sequence[str],
    label_mean: np.ndarray,
    label_std: np.ndarray,
    config: Mapping[str, Any],
    scaler_mean: Optional[np.ndarray] = None,
    scaler_std: Optional[np.ndarray] = None,
) -> Tuple[Path, Path, Path]:
    """Alias for notebook/smoke compatibility."""
    return export_v8_bundle(
        model,
        export_dir,
        device=device,
        window_len=window_len,
        n_features=n_features,
        feature_cols=feature_cols,
        label_mean=label_mean,
        label_std=label_std,
        config=config,
        scaler_mean=scaler_mean,
        scaler_std=scaler_std,
    )

def run_research_training(
    raw_5m: pd.DataFrame,
    *,
    export_dir: Path,
    config: Optional[Mapping[str, Any]] = None,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
) -> Dict[str, Any]:
    """End-to-end research train on a 5m OHLCV frame (immutable)."""
    cfg = default_research_config()
    if config:
        cfg.update(dict(config))
    set_research_seed(int(cfg["seed"]))
    assembled = assemble_raw_frame(raw_5m, funding_df=funding_df, oi_df=oi_df)
    ohlcv_quality_report(assembled, "5m", symbol=str(cfg.get("symbol") or "BTCUSD"))
    labeled = build_labeled_frame(assembled, config=cfg)
    feature_cols = list(v9_feature_cols_for_resolution("5m"))
    feature_cols = [c for c in feature_cols if c in labeled.columns]
    leakage_audit(feature_cols)

    window_len = int(cfg["sequence_length"])
    stride = int(cfg.get("stride") or 4)
    n = len(labeled)
    embargo = int(
        cfg.get("embargo_bars")
        or cfg.get("path_label_horizon_bars")
        or MAX_V8_HORIZON_BARS
    )
    slices = chronological_split(
        n,
        train_ratio=float(cfg["train_ratio"]),
        validation_ratio=float(cfg["validation_ratio"]),
        embargo=embargo,
    )
    train_df = labeled.iloc[slices["train"]].reset_index(drop=True)
    train_x = train_df[feature_cols].to_numpy(dtype=np.float64)
    feature_finite_report(train_x, feature_cols)
    scaler_mean, scaler_std = fit_train_scaler(train_x)
    per_window = str(cfg.get("scaler_mode") or "train_fit") == "per_window"
    packed_all = windows_from_frame(
        labeled,
        feature_cols=feature_cols,
        window_len=window_len,
        stride=stride,
        scaler_mean=None if per_window else scaler_mean,
        scaler_std=None if per_window else scaler_std,
        per_window_zscore=per_window,
    )
    win_slices = chronological_split(
        len(packed_all["x"]),
        train_ratio=float(cfg["train_ratio"]),
        validation_ratio=float(cfg["validation_ratio"]),
        embargo=max(1, min(4, embargo // max(stride, 1))),
    )
    splits = split_window_dict(packed_all, win_slices)
    y_mean, y_std = fit_label_stats(splits["train"]["y_path"])
    dir_w = inverse_frequency_class_weights(
        splits["train"]["horizon_dirs"].reshape(-1),
        NEXT_DIRECTION_CARDINALITY,
    )
    pat_w = inverse_frequency_class_weights(
        splits["train"].get(
            "pattern_ids", np.zeros(len(splits["train"]["x"]), dtype=np.int64)
        ),
        CHART_PATTERN_CARDINALITY,
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}  windows train/val/test="
          f"{len(splits['train']['x'])}/{len(splits['val']['x'])}/{len(splits['test']['x'])}")

    loaders = {}
    for name, shuffle in (("train", True), ("val", False), ("test", False)):
        yz, mask = standardize_labels(splits[name]["y_path"], y_mean, y_std)
        loaders[name] = make_loader(
            splits[name],
            y_path_z=yz,
            path_mask=mask,
            batch_size=int(cfg["batch_size"]),
            shuffle=shuffle,
            per_window_zscore=per_window,
        )

    model = NextCandleTransformer(
        n_features=len(feature_cols),
        d_model=int(cfg.get("d_model") or 64),
        nhead=int(cfg.get("nhead") or 4),
        num_layers=int(cfg.get("num_layers") or 2),
        dropout=float(cfg["dropout"]),
        max_len=window_len,
        n_continuous=len(V8_CONTINUOUS_LABEL_COLS),
    ).to(device)
    train_hist = train_next_candle(
        model,
        loaders["train"],
        loaders["val"],
        device=device,
        epochs=int(cfg["epochs"]),
        lr=float(cfg["learning_rate"]),
        weight_decay=float(cfg["weight_decay"]),
        patience=int(cfg["early_stopping_patience"]),
        loss_weights=cfg.get("loss_weights"),
        dir_class_weights=torch.tensor(dir_w, dtype=torch.float32),
        pattern_class_weights=torch.tensor(pat_w, dtype=torch.float32),
    )
    if not train_hist.get("ok"):
        raise RuntimeError(f"Training aborted with non-finite loss: {train_hist}")
    print("Validation:")
    val_metrics = evaluate_structure_heads(model, loaders["val"], device=device)
    print("Test (untouched):")
    test_metrics = evaluate_structure_heads(model, loaders["test"], device=device)

    if cfg.get("run_ablations"):
        groups = ablation_feature_groups("5m")
        print("Ablation direction accuracy (val, 1 epoch):")
        for key, cols in groups.items():
            idx = np.array(
                [feature_cols.index(c) for c in cols if c in feature_cols],
                dtype=np.int64,
            )
            if len(idx) < 3:
                continue
            acc = run_ablation_epoch(
                splits["train"],
                splits["val"],
                feature_index=idx,
                config=cfg,
                device=device,
                y_mean=y_mean,
                y_std=y_std,
            )
            print(f"  {key}: {acc:.3f}  n_features={len(idx)}")

    onnx_path, cfg_path, meta_path = export_v9_bundle(
        model,
        export_dir,
        device=device,
        window_len=window_len,
        n_features=len(feature_cols),
        feature_cols=feature_cols,
        label_mean=y_mean,
        label_std=y_std,
        config=cfg,
        scaler_mean=None if per_window else scaler_mean,
        scaler_std=None if per_window else scaler_std,
    )
    return {
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "onnx": str(onnx_path),
        "feature_config": str(cfg_path),
        "metadata": str(meta_path),
        "n_features": len(feature_cols),
        "chart_pattern_col": CHART_PATTERN_COL,
    }

def fetch_and_train(
    *,
    export_dir: Path,
    cache_dir: Path,
    config: Optional[Mapping[str, Any]] = None,
    refresh_data: bool = False,
) -> Dict[str, Any]:
    """Fetch 5m history from Delta India (cached) and run research training."""
    cfg = default_research_config()
    if config:
        cfg.update(dict(config))
    cache_dir.mkdir(parents=True, exist_ok=True)
    parquet = cache_dir / "btcusd_5m_raw.parquet"
    if parquet.is_file() and not refresh_data:
        raw = pd.read_parquet(parquet)
    else:
        raw = fetch_history_bundle(
            symbol=str(cfg["symbol"]),
            resolution="5m",
            history_days=int(cfg.get("history_days") or 900),
            base_url=str(cfg.get("base_url") or "https://api.india.delta.exchange"),
        )
        parquet.parent.mkdir(parents=True, exist_ok=True)
        raw.to_parquet(parquet, index=False)
    return run_research_training(raw, export_dir=export_dir, config=cfg)


## 02 CONFIG

In [ ]:
from pathlib import Path
import json

CONFIG = default_research_config()
# Smoke overrides (comment out for a full research run):
# CONFIG["epochs"] = 2
# CONFIG["history_days"] = 120
# CONFIG["sequence_length"] = 64
CONFIG["run_optuna"] = False
CONFIG["run_shap"] = False
CONFIG["run_ablations"] = False
CONFIG["run_walk_forward"] = False

_content = Path("/content")
_root = _content if _content.is_dir() else Path(".")
export_dir = _root / "export" / "JackSparrow_Transformer_BTCUSD_5m"
export_dir.mkdir(parents=True, exist_ok=True)
cache_dir = _root / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, indent=2, default=str))


## 03 Seeds

In [ ]:
set_research_seed(int(CONFIG["seed"]))
print("seed", CONFIG["seed"])


## 04 Load OHLCV

In [ ]:
parquet = cache_dir / "btcusd_5m_raw.parquet"
refresh_data = False
if parquet.is_file() and not refresh_data:
    raw_5m = pd.read_parquet(parquet)
    print(f"Loaded cache {parquet} rows={len(raw_5m)}")
else:
    raw_5m = fetch_history_bundle(
        symbol=str(CONFIG["symbol"]),
        resolution="5m",
        history_days=int(CONFIG.get("history_days") or 900),
        base_url=str(CONFIG.get("base_url") or "https://api.india.delta.exchange"),
    )
    raw_5m.to_parquet(parquet, index=False)
    print(f"Fetched and cached {parquet} rows={len(raw_5m)}")
assert list(raw_5m.columns)
print(raw_5m.head(3))


## 05 Data quality

In [ ]:
quality = ohlcv_quality_report(raw_5m, "5m", symbol=str(CONFIG["symbol"]))
print(json.dumps(quality, indent=2, default=str))


## 06 Causal features

In [ ]:
labeled = build_labeled_frame(raw_5m, config=CONFIG)
feature_cols = [c for c in v9_feature_cols_for_resolution("5m") if c in labeled.columns]
print(f"labeled rows={len(labeled)} n_features={len(feature_cols)}")
print("chart_pattern_id in frame (target only):", CHART_PATTERN_COL in labeled.columns)
print("chart_pattern_id in model inputs:", CHART_PATTERN_COL in feature_cols)
print("chart_pattern_id mix:")
print(labeled[CHART_PATTERN_COL].value_counts().sort_index().to_dict())
print(labeled[feature_cols[:8]].tail(2))


## 07 Leakage audit

In [ ]:
leakage_audit(feature_cols)
print("Leakage audit passed: no t+1 / path / horizon target columns in X.")


## 08 Multi-horizon path labels

In [ ]:
target_cols = [
    *HORIZON_DIR_COLS,
    *HORIZON_STRUCTURE_COLS,
    *V8_CONTINUOUS_LABEL_COLS,
    VOLUME_STATE_COL,
]
present = [c for c in target_cols if c in labeled.columns]
for col in present:
    series = labeled[col].dropna()
    n_unique = int(series.nunique())
    kind = "cont" if n_unique >= 20 else n_unique
    print(f"{col}: n={len(series)} unique={kind}")

print("Direction class mix / |move|/ATR (thresholds 0.5 and 2.0 ATR):")
direction_class_mix_report(labeled)
print("Structure class mix:")
for col in HORIZON_STRUCTURE_COLS:
    if col in labeled.columns:
        print(col, labeled[col].value_counts(dropna=True).sort_index().to_dict())
if "h5m_vol" in labeled.columns:
    print("h5m_vol nunique", int(labeled["h5m_vol"].nunique(dropna=True)))


## 09 Temporal split

In [ ]:
window_len = int(CONFIG["sequence_length"])
stride = int(CONFIG.get("stride") or 4)
embargo = int(
    CONFIG.get("embargo_bars")
    or CONFIG.get("path_label_horizon_bars")
    or MAX_V8_HORIZON_BARS
)
bar_slices = chronological_split(
    len(labeled),
    train_ratio=float(CONFIG["train_ratio"]),
    validation_ratio=float(CONFIG["validation_ratio"]),
    embargo=embargo,
)
print({k: (v.start, v.stop) for k, v in bar_slices.items()})


## 10 Scaler

In [ ]:
train_df = labeled.iloc[bar_slices["train"]].reset_index(drop=True)
train_x = train_df[feature_cols].to_numpy(dtype=np.float64)
feature_finite_report(train_x, feature_cols)
scaler_mean, scaler_std = fit_train_scaler(train_x)
print("scaler fitted on TRAIN rows only", scaler_mean.shape)
per_window = str(CONFIG.get("scaler_mode") or "train_fit") == "per_window"


## 11 Sequence datasets

In [ ]:
packed_all = windows_from_frame(
    labeled,
    feature_cols=feature_cols,
    window_len=window_len,
    stride=stride,
    scaler_mean=None if per_window else scaler_mean,
    scaler_std=None if per_window else scaler_std,
    per_window_zscore=per_window,
)
win_slices = chronological_split(
    len(packed_all["x"]),
    train_ratio=float(CONFIG["train_ratio"]),
    validation_ratio=float(CONFIG["validation_ratio"]),
    embargo=max(1, min(4, embargo // max(stride, 1))),
)
splits = split_window_dict(packed_all, win_slices)
y_mean, y_std = fit_label_stats(splits["train"]["y_path"])
print("windows", {k: len(v["x"]) for k, v in splits.items()})
dir_class_w = inverse_frequency_class_weights(
    splits["train"]["horizon_dirs"].reshape(-1), NEXT_DIRECTION_CARDINALITY
)
pat_class_w = inverse_frequency_class_weights(
    splits["train"]["pattern_ids"], CHART_PATTERN_CARDINALITY
)
print("dir class weights", dir_class_w.round(3).tolist())


## 12 Transformer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NextCandleTransformer(
    n_features=len(feature_cols),
    d_model=int(CONFIG.get("d_model") or 64),
    nhead=int(CONFIG.get("nhead") or 4),
    num_layers=int(CONFIG.get("num_layers") or 2),
    dropout=float(CONFIG["dropout"]),
    max_len=window_len,
    n_continuous=len(V8_CONTINUOUS_LABEL_COLS),
).to(device)
print(model)
print("device", device)


## 13 Multi-task loss

In [ ]:
print("Loss lambdas (structure cannot be zeroed):")
print(json.dumps(CONFIG.get("loss_weights") or {}, indent=2))
print("Heads: 6x 5-class direction, 6x structure, 24-d path, volume, pattern.")


## 14 Train + early stopping

In [ ]:
loaders = {}
for name, shuffle in (("train", True), ("val", False), ("test", False)):
    yz, mask = standardize_labels(splits[name]["y_path"], y_mean, y_std)
    loaders[name] = make_loader(
        splits[name],
        y_path_z=yz,
        path_mask=mask,
        batch_size=int(CONFIG["batch_size"]),
        shuffle=shuffle,
        per_window_zscore=per_window,
    )
train_hist = train_next_candle(
    model,
    loaders["train"],
    loaders["val"],
    device=device,
    epochs=int(CONFIG["epochs"]),
    lr=float(CONFIG["learning_rate"]),
    weight_decay=float(CONFIG["weight_decay"]),
    patience=int(CONFIG["early_stopping_patience"]),
    loss_weights=CONFIG.get("loss_weights"),
    dir_class_weights=torch.tensor(dir_class_w, dtype=torch.float32),
    pattern_class_weights=torch.tensor(pat_class_w, dtype=torch.float32),
)
print(train_hist)
if not train_hist.get("ok"):
    raise RuntimeError(f"Training aborted with non-finite loss: {train_hist}")


## 15 Validation metrics

In [ ]:
print("Validation:")
val_metrics = evaluate_structure_heads(model, loaders["val"], device=device)


## 16 Test

In [ ]:
print("Test (untouched, one run of the frozen model):")
test_metrics = evaluate_structure_heads(model, loaders["test"], device=device)


## 17 Walk-forward

In [ ]:
if CONFIG.get("run_walk_forward"):
    run_walk_forward_eval(
        packed_all, config=CONFIG, device=device, y_mean=y_mean, y_std=y_std
    )
else:
    n_folds = int(CONFIG.get("walk_forward_folds") or 3)
    folds = walk_forward_slices(len(packed_all["x"]), folds=n_folds, embargo=1)
    fold_spans = [(s.start, s.stop, v.start, v.stop) for s, v in folds]
    print(f"Walk-forward folds (not executed): {fold_spans}")


## 18 Horizon confusion

In [ ]:
model.eval()
xb = torch.tensor(splits["val"]["x"][:256], dtype=torch.float32, device=device)
xc = torch.tensor(splits["val"]["x_cat"][:256], dtype=torch.long, device=device)
with torch.no_grad():
    outs = model(xb, xc)
pred_dir = outs[0].argmax(dim=1).cpu().numpy()
true_dir = splits["val"]["horizon_dirs"][: len(pred_dir), 0]
print("h5m direction confusion (true x pred):")
print(confusion_counts(pred_dir, true_dir, NEXT_DIRECTION_CARDINALITY))
pred_h10 = outs[1].argmax(dim=1).cpu().numpy()
true_h10 = splits["val"]["horizon_dirs"][: len(pred_h10), 1]
print("h10m direction confusion (true x pred):")
print(confusion_counts(pred_h10, true_h10, NEXT_DIRECTION_CARDINALITY))


## 19 Pattern × context

In [ ]:
pattern_context_table(labeled)


## 20 SHAP

In [ ]:
shap_grouped_stub(feature_cols, enabled=bool(CONFIG.get("run_shap")))


## 21 Ablations A–F

In [ ]:
if CONFIG.get("run_ablations"):
    groups = ablation_feature_groups("5m")
    print("Ablation direction accuracy (val, 1 epoch, out-of-sample splits):")
    for key, cols in groups.items():
        idx = np.array([feature_cols.index(c) for c in cols if c in feature_cols], dtype=np.int64)
        if len(idx) < 3:
            continue
        acc = run_ablation_epoch(
            splits["train"], splits["val"], feature_index=idx,
            config=CONFIG, device=device, y_mean=y_mean, y_std=y_std,
        )
        print(f"  {key}: {acc:.3f}  n_features={len(idx)}")
    print("If F wins only on train, call overfitting — report val/test only.")
else:
    print("Ablations skipped (CONFIG run_ablations=False). Groups A–F: OHLCV → +geometry → +trend → +structure → +chart → +HTF.")


## 22 Optuna

In [ ]:
CONFIG = optuna_search_stub(CONFIG)


## 23 Re-train

In [ ]:
print("Best CONFIG already trained on the research train split with val early stopping.")
print("To re-train on train+val after Optuna, concatenate those loaders and call train_next_candle again.")
print("Never peek at the final test split during search.")


## 24 Final untouched test

In [ ]:
print("Final untouched test (frozen weights):")
final_test = evaluate_structure_heads(model, loaders["test"], device=device)
print(final_test)


## 25 Save

In [ ]:
if not train_hist.get("ok"):
    print("Skip save: training did not succeed", train_hist)
else:
    artifact = {
        "feature_cols": list(feature_cols),
        "loss_weights": CONFIG.get("loss_weights"),
        "seed": CONFIG.get("seed"),
        "sequence_length": window_len,
        "scaler_mode": CONFIG.get("scaler_mode"),
        "feature_contract_version": FEATURE_CONTRACT_VERSION,
        "val_metrics": val_metrics,
        "test_metrics": final_test,
    }
    (export_dir / "research_run.json").write_text(
        json.dumps(artifact, indent=2, default=str), encoding="utf-8"
    )
    print("Wrote", export_dir / "research_run.json")


## 26 JackSparrow export

In [ ]:
if not train_hist.get("ok"):
    print("Skip ONNX export: training did not succeed", train_hist)
else:
    onnx_path, cfg_path, meta_path = export_v9_bundle(
        model,
        export_dir,
        device=device,
        window_len=window_len,
        n_features=len(feature_cols),
        feature_cols=feature_cols,
        label_mean=y_mean,
        label_std=y_std,
        config=CONFIG,
        scaler_mean=None if per_window else scaler_mean,
        scaler_std=None if per_window else scaler_std,
    )
    print("Exported", onnx_path)
    print("feature_config", cfg_path)
    print("metadata", meta_path)
    print("Copy this directory into agent/model_storage/ after validate_transformer_bundle.py")


## Notes

- Historical OHLCV is **immutable**. Training updates **weights only**.
- Scaler is fit on **train windows/rows only**. Optuna/walk-forward never see the final test set.
- Named candlestick class is an **input embedding**. Movement value is per-horizon
  **MFE/MAE / path_edge** at 5m through 2h. ``h5m_vol`` is RMS |log return|
  (not std of a 1-sample window). Structure labels use the **terminal** bar
  of each horizon so 1h/2h do not collapse to BREAKOUT.
- Production v6 all-TF notebook remains the live 15m–2h trainer until this v9
  5m export is proven.
- Agent follow-on: 15m still owns setup SL/TP; 5m timing from 5m+10m
  P(UP)-P(DOWN); 15m–2h packets on the 5m model are telemetry only.
- Promote only if test balanced direction is above chance at 5m and 10m and
  path_vol_corr stays above 0.15. Raw accuracy climbing with horizon is not enough.
